<a href="https://colab.research.google.com/github/Ahmadlazz/data_acquisition/blob/main/UTS_Data_Acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1




In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

import os
import time
import hashlib
import re

from datetime import datetime
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

2

In [ ]:
# Membuat folder penyimpanan
os.makedirs("data/raw/download", exist_ok=True)
os.makedirs("data/raw/scraping", exist_ok=True)
os.makedirs("data/process", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# Masukkan email kampus sendiri
EMAIL_KAMPUS = "GANTI_DENGAN_EMAIL_KAMPUS"

# Identitas request
HEADERS = {
    "User-Agent": f"UTS-Data-Acquisition/1.0 ({EMAIL_KAMPUS})"
}

# Pengaturan request
TIMEOUT = 20
DELAY = 2

print("Persiapan proyek berhasil")

Persiapan proyek berhasil


## Akuisisi Dataset Terbuka

Dataset diperoleh dari portal data pemerintah
yang berkaitan dengan sektor pariwisata Bali.

In [ ]:
# Melihat file yang tersedia
os.listdir("data/raw/download")

['Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv']

3

In [ ]:
# Mengambil nama file CSV dari folder
file_dataset = [
    file for file in os.listdir("data/raw/download")
    if file.endswith(".csv")
]

# Memastikan file tersedia
if not file_dataset:
    raise FileNotFoundError("File CSV belum ditemukan")

# Membaca dataset
lokasi_file = "data/raw/download/" + file_dataset[0]

df_download = pd.read_csv(lokasi_file)

# Menampilkan lima data pertama
df_download.head()

,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497,487449,554665,560434,659065,707476,819496,826446,777537,707584,656709,731293,7972651
4,Ngurah Rai,329909,317005,366956,410281,439454,478127,541272,522063,508297,458845,397522,478382,5248113


4

In [ ]:
# Jumlah baris dan kolom
print("Jumlah baris:", df_download.shape[0])
print("Jumlah kolom:", df_download.shape[1])

# Informasi tipe data
df_download.info()

# Statistik data
display(df_download.describe(include="all"))

# Melihat nama kolom
print(df_download.columns.tolist())

Jumlah baris: 38
Jumlah kolom: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pintu Masuk  35 non-null     object
 1   Unnamed: 1   38 non-null     object
 2   Unnamed: 2   36 non-null     object
 3   Unnamed: 3   36 non-null     object
 4   Unnamed: 4   36 non-null     object
 5   Unnamed: 5   36 non-null     object
 6   Unnamed: 6   36 non-null     object
 7   Unnamed: 7   36 non-null     object
 8   Unnamed: 8   36 non-null     object
 9   Unnamed: 9   36 non-null     object
 10  Unnamed: 10  36 non-null     object
 11  Unnamed: 11  36 non-null     object
 12  Unnamed: 12  36 non-null     object
 13  Unnamed: 13  36 non-null     object
dtypes: object(14)
memory usage: 4.3+ KB


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
count,35,38,36,36,36,36,36,36,36,36,36,36,36,36
unique,35,34,32,32,31,32,32,33,32,32,31,32,33,34
top,A. Pintu Udara,0,1,0,0,0,0,0,0,0,0,0,0,-
freq,1,3,3,2,4,3,3,2,3,3,4,3,2,2


['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


5

In [ ]:
if len(df_download) >= 100:
    print("Dataset memenuhi minimal 100 baris")
else:
    print("Dataset belum memenuhi minimal 100 baris")

Dataset belum memenuhi minimal 100 baris


6

In [ ]:
# Menghitung ukuran file dalam byte
ukuran_byte = os.path.getsize(lokasi_file)

# Mengubah byte menjadi MB
ukuran_mb = ukuran_byte / (1024 ** 2)

# Estimasi kebutuhan selama 12 bulan
estimasi_tahunan = ukuran_mb * 12

# Menambahkan cadangan 30 persen
estimasi_total = estimasi_tahunan * 1.30

print("Ukuran file:", round(ukuran_mb, 4), "MB")
print("Estimasi 1 tahun:", round(estimasi_tahunan, 4), "MB")
print("Dengan cadangan 30%:", round(estimasi_total, 4), "MB")

Ukuran file: 0.0029 MB
Estimasi 1 tahun: 0.0351 MB
Dengan cadangan 30%: 0.0456 MB


## Pemeriksaan Etika Data

Pemeriksaan dilakukan terhadap:
1. Lisensi dataset.
2. Ketentuan penggunaan portal.
3. Data pribadi atau PII.
4. Quasi-identifier.
5. Kesesuaian penggunaan data dengan tujuan proyek.

Bukti pemeriksaan dicatat dalam
data_ethics_checklist.md.

7

In [ ]:
# Menghitung data kosong
print(df_download.isnull().sum())

# Memeriksa nama kolom yang mungkin mengandung data pribadi
kata_pii = [
    "nama", "email", "telepon",
    "phone", "alamat", "nik",
    "identitas"
]

kolom_pii = [
    kolom for kolom in df_download.columns
    if any(kata in kolom.lower() for kata in kata_pii)
]

print("Kolom yang perlu diperiksa:", kolom_pii)

Pintu Masuk    3
Unnamed: 1     0
Unnamed: 2     2
Unnamed: 3     2
Unnamed: 4     2
Unnamed: 5     2
Unnamed: 6     2
Unnamed: 7     2
Unnamed: 8     2
Unnamed: 9     2
Unnamed: 10    2
Unnamed: 11    2
Unnamed: 12    2
Unnamed: 13    2
dtype: int64
Kolom yang perlu diperiksa: []


8

In [ ]:
URL_TARGET = (
    "https://www.daybalitour.com/"
    "best-bali-day-tour-packages/"
)

URL_ROBOTS = "https://www.daybalitour.com/robots.txt"

# Mengambil robots.txt
response_robots = requests.get(
    URL_ROBOTS,
    headers=HEADERS,
    timeout=TIMEOUT
)

print("Status:", response_robots.status_code)
print(response_robots.text)

Status: 200
User-agent: *
Disallow: /wp-admin/
Allow: /wp-admin/admin-ajax.php
Disallow: /wp-content/uploads/wpforms/

Sitemap: https://www.daybalitour.com/sitemap.xml
Sitemap: https://www.daybalitour.com/sitemap.rss



9

10

In [ ]:
robot_parser = RobotFileParser()
robot_parser.set_url(URL_ROBOTS)

robot_parser.read()

boleh_scraping = robot_parser.can_fetch(
    HEADERS["User-Agent"],
    URL_TARGET
)

print("Boleh menurut robots.txt:", boleh_scraping)

Boleh menurut robots.txt: True


In [ ]:
if not boleh_scraping:
    raise PermissionError(
        "Halaman tidak diizinkan menurut robots.txt"
    )

# Mengambil halaman
response = requests.get(
    URL_TARGET,
    headers=HEADERS,
    timeout=TIMEOUT
)

# Memeriksa status respons
print("Status HTTP:", response.status_code)

response.raise_for_status()

# Memberikan jeda
time.sleep(DELAY)

print("Halaman berhasil diakses")

Status HTTP: 200
Halaman berhasil diakses


11

In [ ]:
# Menyimpan HTML asli
file_html = "data/raw/scraping/day_bali_tour.html"

with open(file_html, "w", encoding="utf-8") as file:
    file.write(response.text)

print("HTML mentah berhasil disimpan")

HTML mentah berhasil disimpan


12

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# Menampilkan judul halaman
print(soup.title.get_text(strip=True))

# Menghitung jumlah heading
print("Jumlah heading:", len(soup.find_all(["h2", "h3", "h4"])))

Best Bali Day Tour Packages - Day Bali Tour
Jumlah heading: 22


13

In [ ]:
hasil_scraping = []

# Mencari heading paket wisata
for heading in soup.find_all(["h2", "h3", "h4"]):

    nama_paket = heading.get_text(" ", strip=True)

    # Mencari elemen sekitar heading
    parent = heading.parent

    teks_parent = parent.get_text(
        " ",
        strip=True
    )

    # Mencari harga dalam teks
    harga = re.search(
        r"(?:IDR|Rp)\s?[\d.,]+",
        teks_parent,
        re.IGNORECASE
    )

    # Menyimpan hasil
    hasil_scraping.append({
        "nama_paket": nama_paket,
        "harga": harga.group(0) if harga else None,
        "informasi": teks_parent
    })

# Membuat DataFrame
df_scraping = pd.DataFrame(hasil_scraping)

display(df_scraping.head(10))

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...
5,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
6,From IDR 745.000,IDR 745.000,From IDR 745.000 What’s Included: Private Hote...
7,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
8,From IDR 790.000,IDR 790.000,From IDR 790.000 What’s Included: Private Hote...
9,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...


14

In [ ]:
# Menghapus baris yang seluruh kolomnya kosong
df_scraping = df_scraping.dropna(how="all")

# Menghapus baris duplikat
df_scraping = df_scraping.drop_duplicates()

# Menghapus spasi berlebih pada teks
for kolom in df_scraping.select_dtypes(include="object"):
    df_scraping[kolom] = df_scraping[kolom].str.strip()

# Mengatur ulang nomor indeks
df_scraping = df_scraping.reset_index(drop=True)

display(df_scraping)

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...
5,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
6,From IDR 745.000,IDR 745.000,From IDR 745.000 What’s Included: Private Hote...
7,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
8,From IDR 790.000,IDR 790.000,From IDR 790.000 What’s Included: Private Hote...
9,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...


15

In [ ]:
# Menyimpan hasil ke CSV
df_scraping.to_csv(
    "data/process/hasil_scraping.csv",
    index=False
)

# Menyimpan hasil ke JSON
df_scraping.to_json(
    "data/process/hasil_scraping.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("Hasil scraping berhasil disimpan")

Hasil scraping berhasil disimpan


In [ ]:
print("Kolom dataset unduhan:")
print(df_download.columns.tolist())

print("\nKolom data scraping:")
print(df_scraping.columns.tolist())

Kolom dataset unduhan:
['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Kolom data scraping:
['nama_paket', 'harga', 'informasi']


16

In [ ]:
def hitung_md5(lokasi_file):

    md5 = hashlib.md5()

    with open(lokasi_file, "rb") as file:

        for bagian in iter(
            lambda: file.read(4096),
            b""
        ):
            md5.update(bagian)

    return md5.hexdigest()

17

In [ ]:
print(
    hitung_md5(
        "data/raw/scraping/day_bali_tour.html"
    )
)

b27ca0b9e23f86169d62b0a869ab37bd


In [ ]:
# Membuat nama kolom yang benar

nama_kolom = [
    "Pintu Masuk",
    "Januari",
    "Februari",
    "Maret",
    "April",
    "Mei",
    "Juni",
    "Juli",
    "Agustus",
    "September",
    "Oktober",
    "November",
    "Desember",
    "Tahunan"
]

# Mengambil data mulai baris ke-4
df_download_bersih = df_download.iloc[3:].copy()

# Mengganti nama kolom
df_download_bersih.columns = nama_kolom

# Menghapus baris yang tidak memiliki nama pintu masuk
df_download_bersih = df_download_bersih.dropna(
    subset=["Pintu Masuk"]
)

# Merapikan nama pintu masuk
df_download_bersih["Pintu Masuk"] = (
    df_download_bersih["Pintu Masuk"].astype(str).str.strip()
)

# Mengubah data jumlah kunjungan menjadi angka
kolom_angka = nama_kolom[1:]

for kolom in kolom_angka:
    df_download_bersih[kolom] = pd.to_numeric(
        df_download_bersih[kolom],
        errors="coerce"
    )

# Menampilkan hasil
display(df_download_bersih.head(10))

print("Jumlah baris:", len(df_download_bersih))
print("Jumlah kolom:", len(df_download_bersih.columns))

,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497.0,487449.0,554665.0,560434.0,659065.0,707476.0,819496.0,826446.0,777537.0,707584.0,656709.0,731293.0,7972651.0
4,Ngurah Rai,329909.0,317005.0,366956.0,410281.0,439454.0,478127.0,541272.0,522063.0,508297.0,458845.0,397522.0,478382.0,5248113.0
5,Soekarno-Hatta,115101.0,124803.0,140859.0,99289.0,158550.0,161114.0,204000.0,218512.0,190058.0,181627.0,186695.0,172397.0,1953005.0
6,Juanda,10777.0,11882.0,10606.0,12361.0,15734.0,18025.0,20303.0,25548.0,25593.0,21996.0,22389.0,23244.0,218458.0
7,Kualanamu,14146.0,15072.0,16106.0,15497.0,16262.0,16527.0,17478.0,20843.0,15783.0,14161.0,16363.0,18237.0,196475.0
8,Husein Sastranegara,0.0,0.0,0.0,0.0,17.0,24.0,3.0,0.0,0.0,0.0,0.0,0.0,44.0
9,Adi Sucipto,3878.0,4839.0,5007.0,5592.0,8160.0,11188.0,12444.0,12974.0,11855.0,8536.0,9675.0,11338.0,105486.0
10,Bandara Int. Lombok,2202.0,2355.0,2099.0,2903.0,4848.0,5721.0,6433.0,6245.0,6294.0,5904.0,5278.0,7304.0,57586.0
11,Sam Ratulangi,803.0,1848.0,4793.0,4156.0,3798.0,3691.0,3782.0,5397.0,5922.0,4645.0,4611.0,3917.0,47363.0
12,Minangkabau,2878.0,3590.0,2765.0,4100.0,4624.0,5226.0,4946.0,5577.0,5453.0,4925.0,5898.0,6710.0,56692.0


Jumlah baris: 35
Jumlah kolom: 14


18

In [ ]:
# Membuat daftar informasi sumber
daftar_provenance = []

file_sumber = [
    "data/raw/scraping/day_bali_tour.html"
]

for file in file_sumber:

    ukuran = os.path.getsize(file)

    informasi = {
        "nama_file": os.path.basename(file),
        "ukuran_byte": ukuran,
        "md5": hitung_md5(file),
        "tanggal_akses": datetime.now().isoformat()
    }

    daftar_provenance.append(informasi)

# Membuat tabel provenance
df_provenance = pd.DataFrame(daftar_provenance)

display(df_provenance)

,nama_file,ukuran_byte,md5,tanggal_akses
0,day_bali_tour.html,391767,b27ca0b9e23f86169d62b0a869ab37bd,2026-11-25T05:08:55.631879


10

In [ ]:
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

print("Provenance berhasil disimpan")

Provenance berhasil disimpan


In [ ]:
for folder, subfolder, files in os.walk("."):

    if ".ipynb_checkpoints" in folder:
        continue

    print("\nFolder:", folder)

    for nama_file in files:
        print("-", nama_file)


Folder: .

Folder: ./.config
- default_configs.db
- gce
- .last_update_check.json
- hidden_gcloud_config_universe_descriptor_data_cache_configs.db
- config_sentinel
- active_config
- .last_survey_prompt.yaml
- .last_opt_in_prompt.yaml

Folder: ./.config/logs

Folder: ./.config/logs/2026.10.26
- 22.18.15.518759.log
- 22.18.38.571729.log
- 22.18.53.588181.log
- 22.18.45.880512.log
- 22.18.52.867204.log
- 22.18.44.768076.log

Folder: ./.config/configurations
- config_default

Folder: ./docs
- provenance.csv

Folder: ./drive

Folder: ./drive/.shortcut-targets-by-id

Folder: ./drive/MyDrive
- IMG_20250819_194854.jpg
- IMG_20250820_114605 (1).jpg
- IMG_20250820_114605.jpg
- IMG_20250820_122118.jpg
- IMG_20250820_114615.jpg
- 20250822_102519AMByGPSMapCamera.jpg
- 20250823_103512AMByGPSMapCamera.jpg
- IMG_20250823_134016.jpg
- 20250825_84339AMByGPSMapCamera.jpg
- IMG_20250825_101652.jpg
- 20250826_101530AMByGPSMapCamera.jpg
- 20250827_103831AMByGPSMapCamera.jpg
- 20250827_103930AMByGPSMapCame

In [ ]:
import os

folder = "data/raw/download"

print("Daftar file dalam folder download:")

for file in os.listdir(folder):
    print("-", file)

Daftar file dalam folder download:
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv


In [ ]:
URL_SCRAPING = "https://www.daybalitour.com/best-bali-day-tour-packages/"

HEADERS = {
    "User-Agent": "Ahmad Paesoel - ahmadpaesoel34@gmail.com"
}

TIMEOUT = 20
DELAY = 2

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

time.sleep(DELAY)

response = requests.get(
    URL_SCRAPING,
    headers=HEADERS,
    timeout=TIMEOUT
)

print("Status Code:", response.status_code)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Judul halaman:", soup.title.get_text(strip=True))

for heading in soup.find_all(["h2", "h3", "h4"]):
    print(heading.name, ":", heading.get_text(" ", strip=True))

Status Code: 200
Judul halaman: Best Bali Day Tour Packages - Day Bali Tour
h2 : Bali Day Tours Highlights
h3 : Bali Waterfall and Best Ubud Tour Packages
h4 : From IDR 580.000
h3 : Bali Best Nusa Penida Island Tour Packages
h4 : From IDR 760.000
h3 : Bali Lempuyang Gate of Heaven Tour Package
h4 : From IDR 745.000
h3 : Bali Ulun Danu and Tanah Lot Tour Package
h4 : From IDR 790.000
h3 : Bali Tanjung Benoa & Uluwatu Tour Package
h4 : From IDR 870.000
h3 : Bali Kintamani Volcano View Tour Package
h4 : From IDR 715.000
h4 : Special Offer
h4 : Bali Activities
h2 : Bali Driver (Tour with Your Itinerary)
h3 : Bali Driver with Toyota Avanza
h4 : From 375.000/Car
h3 : Bali Driver with Toyota Innova
h4 : IDR 1.125.000/Car
h3 : Bali Driver with Toyota Hiace
h4 : IDR 1.275.000/Car


In [ ]:
# Menampilkan seluruh output tanpa terpotong

from IPython.display import display

for heading in soup.find_all(["h2", "h3", "h4"]):
    print(heading.name, ":", heading.get_text(" ", strip=True))

h2 : Bali Day Tours Highlights
h3 : Bali Waterfall and Best Ubud Tour Packages
h4 : From IDR 580.000
h3 : Bali Best Nusa Penida Island Tour Packages
h4 : From IDR 760.000
h3 : Bali Lempuyang Gate of Heaven Tour Package
h4 : From IDR 745.000
h3 : Bali Ulun Danu and Tanah Lot Tour Package
h4 : From IDR 790.000
h3 : Bali Tanjung Benoa & Uluwatu Tour Package
h4 : From IDR 870.000
h3 : Bali Kintamani Volcano View Tour Package
h4 : From IDR 715.000
h4 : Special Offer
h4 : Bali Activities
h2 : Bali Driver (Tour with Your Itinerary)
h3 : Bali Driver with Toyota Avanza
h4 : From 375.000/Car
h3 : Bali Driver with Toyota Innova
h4 : IDR 1.125.000/Car
h3 : Bali Driver with Toyota Hiace
h4 : IDR 1.275.000/Car


In [ ]:
print("Kolom dataset download:")
print(df_download.columns.tolist())

print("\nKolom dataset scraping:")
print(df_scraping.columns.tolist())

Kolom dataset download:
['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Kolom dataset scraping:
['nama_paket', 'harga', 'informasi']


In [ ]:
print("BENTUK DATASET DOWNLOAD:")
print(df_download.shape)

print("\n10 BARIS PERTAMA DATASET DOWNLOAD:")
display(df_download.head(10))

print("\nINFORMASI DATASET DOWNLOAD:")
df_download.info()

BENTUK DATASET DOWNLOAD:
(38, 14)

10 BARIS PERTAMA DATASET DOWNLOAD:


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497,487449,554665,560434,659065,707476,819496,826446,777537,707584,656709,731293,7972651
4,Ngurah Rai,329909,317005,366956,410281,439454,478127,541272,522063,508297,458845,397522,478382,5248113
5,Soekarno-Hatta,115101,124803,140859,99289,158550,161114,204000,218512,190058,181627,186695,172397,1953005
6,Juanda,10777,11882,10606,12361,15734,18025,20303,25548,25593,21996,22389,23244,218458
7,Kualanamu,14146,15072,16106,15497,16262,16527,17478,20843,15783,14161,16363,18237,196475
8,Husein Sastranegara,0,0,0,0,17,24,3,0,0,0,0,0,44
9,Adi Sucipto,3878,4839,5007,5592,8160,11188,12444,12974,11855,8536,9675,11338,105486



INFORMASI DATASET DOWNLOAD:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pintu Masuk  35 non-null     object
 1   Unnamed: 1   38 non-null     object
 2   Unnamed: 2   36 non-null     object
 3   Unnamed: 3   36 non-null     object
 4   Unnamed: 4   36 non-null     object
 5   Unnamed: 5   36 non-null     object
 6   Unnamed: 6   36 non-null     object
 7   Unnamed: 7   36 non-null     object
 8   Unnamed: 8   36 non-null     object
 9   Unnamed: 9   36 non-null     object
 10  Unnamed: 10  36 non-null     object
 11  Unnamed: 11  36 non-null     object
 12  Unnamed: 12  36 non-null     object
 13  Unnamed: 13  36 non-null     object
dtypes: object(14)
memory usage: 4.3+ KB


In [ ]:
print("Jumlah baris:", len(df_download_bersih))
print("Jumlah kolom:", len(df_download_bersih.columns))

print("\nNama kolom:")
print(df_download_bersih.columns.tolist())

print("\nData setelah dibersihkan:")
display(df_download_bersih.head(10))

print("\nTipe data:")
df_download_bersih.info()

Jumlah baris: 35
Jumlah kolom: 14

Nama kolom:
['Pintu Masuk', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember', 'Tahunan']

Data setelah dibersihkan:


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497.0,487449.0,554665.0,560434.0,659065.0,707476.0,819496.0,826446.0,777537.0,707584.0,656709.0,731293.0,7972651.0
4,Ngurah Rai,329909.0,317005.0,366956.0,410281.0,439454.0,478127.0,541272.0,522063.0,508297.0,458845.0,397522.0,478382.0,5248113.0
5,Soekarno-Hatta,115101.0,124803.0,140859.0,99289.0,158550.0,161114.0,204000.0,218512.0,190058.0,181627.0,186695.0,172397.0,1953005.0
6,Juanda,10777.0,11882.0,10606.0,12361.0,15734.0,18025.0,20303.0,25548.0,25593.0,21996.0,22389.0,23244.0,218458.0
7,Kualanamu,14146.0,15072.0,16106.0,15497.0,16262.0,16527.0,17478.0,20843.0,15783.0,14161.0,16363.0,18237.0,196475.0
8,Husein Sastranegara,0.0,0.0,0.0,0.0,17.0,24.0,3.0,0.0,0.0,0.0,0.0,0.0,44.0
9,Adi Sucipto,3878.0,4839.0,5007.0,5592.0,8160.0,11188.0,12444.0,12974.0,11855.0,8536.0,9675.0,11338.0,105486.0
10,Bandara Int. Lombok,2202.0,2355.0,2099.0,2903.0,4848.0,5721.0,6433.0,6245.0,6294.0,5904.0,5278.0,7304.0,57586.0
11,Sam Ratulangi,803.0,1848.0,4793.0,4156.0,3798.0,3691.0,3782.0,5397.0,5922.0,4645.0,4611.0,3917.0,47363.0
12,Minangkabau,2878.0,3590.0,2765.0,4100.0,4624.0,5226.0,4946.0,5577.0,5453.0,4925.0,5898.0,6710.0,56692.0



Tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 3 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Pintu Masuk  35 non-null     object 
 1   Januari      33 non-null     float64
 2   Februari     33 non-null     float64
 3   Maret        33 non-null     float64
 4   April        33 non-null     float64
 5   Mei          33 non-null     float64
 6   Juni         33 non-null     float64
 7   Juli         33 non-null     float64
 8   Agustus      33 non-null     float64
 9   September    33 non-null     float64
 10  Oktober      33 non-null     float64
 11  November     33 non-null     float64
 12  Desember     33 non-null     float64
 13  Tahunan      33 non-null     float64
dtypes: float64(13), object(1)
memory usage: 4.0+ KB


In [ ]:
import os

folder_data = "data/raw/download"

print("Daftar file CSV:\n")

for file in os.listdir(folder_data):
    if file.lower().endswith(".csv"):
        print(file)

Daftar file CSV:

Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv


In [ ]:
import pandas as pd
import os

folder_data = "data/raw/download"

file_csv = [
    file for file in os.listdir(folder_data)
    if file.lower().endswith(".csv")
]

print("Jumlah dataset:", len(file_csv))

for file in sorted(file_csv):
    print("-", file)

Jumlah dataset: 4
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv


In [ ]:
# Membaca seluruh dataset berdasarkan tahun

dataset_tahunan = {}

for tahun in [2022, 2023, 2024, 2025]:

    nama_file = (
        f"Jumlah Kunjungan Wisatawan Mancanegara per bulan "
        f"ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, {tahun}.csv"
    )

    lokasi_file = os.path.join(folder_data, nama_file)

    dataset_tahunan[tahun] = pd.read_csv(lokasi_file)

    print(f"Dataset {tahun} berhasil dibaca")

Dataset 2022 berhasil dibaca
Dataset 2023 berhasil dibaca
Dataset 2024 berhasil dibaca
Dataset 2025 berhasil dibaca


In [ ]:
for tahun, data in dataset_tahunan.items():

    print(f"\n===== DATA TAHUN {tahun} =====")
    print("Jumlah baris:", len(data))
    print("Jumlah kolom:", len(data.columns))

    display(data.head(5))


===== DATA TAHUN 2022 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555,17834,39062,97401,183830,280482,387184,422500,424198,447074,429979,551927,3296026
4,Ngurah Rai,0,1293,14617,58315,115553,181545,246442,276627,291115,305152,287025,376361,2154045



===== DATA TAHUN 2023 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497,487449,554665,560434,659065,707476,819496,826446,777537,707584,656709,731293,7972651
4,Ngurah Rai,329909,317005,366956,410281,439454,478127,541272,522063,508297,458845,397522,478382,5248113



===== DATA TAHUN 2024 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,631961,735072,682439,750085,835722,821654,994583,987547,941903,859747,756972,832865,9830550
4,Ngurah Rai,414937,454398,463804,502870,544492,520792,625569,613540,591848,554892,472714,548685,6308541



===== DATA TAHUN 2025 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291
4,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585


In [ ]:
# Daftar nama kolom dataset

nama_kolom = [
    "Pintu Masuk",
    "Januari",
    "Februari",
    "Maret",
    "April",
    "Mei",
    "Juni",
    "Juli",
    "Agustus",
    "September",
    "Oktober",
    "November",
    "Desember",
    "Tahunan"
]


# Fungsi untuk membersihkan dataset

def bersihkan_dataset(data, tahun):

    # Mengambil data mulai baris ke-4
    data_bersih = data.iloc[3:].copy()

    # Memberikan nama kolom
    data_bersih.columns = nama_kolom

    # Menghapus baris tanpa nama pintu masuk
    data_bersih = data_bersih.dropna(
        subset=["Pintu Masuk"]
    )

    # Merapikan nama pintu masuk
    data_bersih["Pintu Masuk"] = (
        data_bersih["Pintu Masuk"]
        .astype(str)
        .str.strip()
    )

    # Mengubah kolom jumlah kunjungan menjadi angka
    kolom_angka = nama_kolom[1:]

    for kolom in kolom_angka:
        data_bersih[kolom] = pd.to_numeric(
            data_bersih[kolom],
            errors="coerce"
        )

    # Menambahkan identitas tahun
    data_bersih["Tahun"] = tahun

    return data_bersih


# Membersihkan empat dataset

dataset_bersih = {}

for tahun, data in dataset_tahunan.items():

    dataset_bersih[tahun] = bersihkan_dataset(
        data,
        tahun
    )

    print(f"Dataset {tahun} berhasil dibersihkan!")

Dataset 2022 berhasil dibersihkan!
Dataset 2023 berhasil dibersihkan!
Dataset 2024 berhasil dibersihkan!
Dataset 2025 berhasil dibersihkan!


In [ ]:
for tahun, data in dataset_bersih.items():

    print(f"\n===== DATA TAHUN {tahun} =====")

    print("Jumlah baris:", len(data))
    print("Jumlah kolom:", len(data.columns))

    display(data.head(5))


===== DATA TAHUN 2022 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0,2022
4,Ngurah Rai,0.0,1293.0,14617.0,58315.0,115553.0,181545.0,246442.0,276627.0,291115.0,305152.0,287025.0,376361.0,2154045.0,2022
5,Soekarno-Hatta,14073.0,15406.0,23320.0,36061.0,57844.0,85587.0,119179.0,121427.0,111033.0,112527.0,113729.0,124475.0,934661.0,2022
6,Juanda,0.0,0.0,57.0,1468.0,4598.0,4722.0,6087.0,8386.0,9152.0,9415.0,9644.0,14264.0,67793.0,2022
7,Kualanamu,0.0,0.0,11.0,494.0,3861.0,5165.0,7518.0,10287.0,5895.0,11454.0,10615.0,19198.0,74498.0,2022



===== DATA TAHUN 2023 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,484497.0,487449.0,554665.0,560434.0,659065.0,707476.0,819496.0,826446.0,777537.0,707584.0,656709.0,731293.0,7972651.0,2023
4,Ngurah Rai,329909.0,317005.0,366956.0,410281.0,439454.0,478127.0,541272.0,522063.0,508297.0,458845.0,397522.0,478382.0,5248113.0,2023
5,Soekarno-Hatta,115101.0,124803.0,140859.0,99289.0,158550.0,161114.0,204000.0,218512.0,190058.0,181627.0,186695.0,172397.0,1953005.0,2023
6,Juanda,10777.0,11882.0,10606.0,12361.0,15734.0,18025.0,20303.0,25548.0,25593.0,21996.0,22389.0,23244.0,218458.0,2023
7,Kualanamu,14146.0,15072.0,16106.0,15497.0,16262.0,16527.0,17478.0,20843.0,15783.0,14161.0,16363.0,18237.0,196475.0,2023



===== DATA TAHUN 2024 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,631961.0,735072.0,682439.0,750085.0,835722.0,821654.0,994583.0,987547.0,941903.0,859747.0,756972.0,832865,9830550,2024
4,Ngurah Rai,414937.0,454398.0,463804.0,502870.0,544492.0,520792.0,625569.0,613540.0,591848.0,554892.0,472714.0,548685,6308541,2024
5,Soekarno-Hatta,157979.0,187641.0,160917.0,157409.0,201574.0,225985.0,271680.0,272849.0,250852.0,218170.0,212178.0,207019,2524253,2024
6,Juanda,17196.0,28026.0,18543.0,29043.0,31222.0,25015.0,34446.0,38587.0,31700.0,28182.0,20820.0,19265,322045,2024
7,Kualanamu,14751.0,24628.0,15068.0,25178.0,20469.0,16949.0,22693.0,21930.0,24956.0,20129.0,19145.0,21142,247038,2024



===== DATA TAHUN 2025 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291,2025
4,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585,2025
5,Soekarno-Hatta,193433,199306,146390,186523,233751,236013,290369,304970,244404,254968,240330,230381,2760838,2025
6,Juanda,22119,19205,15647,24800,33409,34092,34421,37406,32664,29534,21629,25019,329945,2025
7,Kualanamu,23011,22546,16881,19709,26183,25183,28306,29858,29107,24621,20826,23425,289656,2025


In [ ]:
# Menggabungkan empat dataset tahunan

df_download_gabungan = pd.concat(
    dataset_bersih.values(),
    ignore_index=True
)

# Mengurutkan berdasarkan tahun dan pintu masuk
df_download_gabungan = df_download_gabungan.sort_values(
    by=["Tahun", "Pintu Masuk"]
).reset_index(drop=True)

print("Dataset berhasil digabungkan!")
print("Jumlah baris:", len(df_download_gabungan))
print("Jumlah kolom:", len(df_download_gabungan.columns))

display(df_download_gabungan.head(10))

Dataset berhasil digabungkan!
Jumlah baris: 140
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
0,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0,2022
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
5,B. Pintu Laut,58442.0,51539.0,60357.0,69085.0,105719.0,134664.0,184193.0,164918.0,196668.0,207390.0,193894.0,299967.0,1726836.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
8,C. Pintu Darat,63301.0,48136.0,59210.0,63590.0,65371.0,68737.0,73744.0,83091.0,79740.0,79764.0,80910.0,100575.0,866169.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022


In [ ]:
print("Jumlah data duplikat:")
print(df_download_gabungan.duplicated().sum())

print("\nJumlah data berdasarkan tahun:")
print(df_download_gabungan["Tahun"].value_counts().sort_index())

Jumlah data duplikat:
0

Jumlah data berdasarkan tahun:
Tahun
2022    35
2023    35
2024    35
2025    35
Name: count, dtype: int64


In [ ]:
df_download_gabungan.to_csv(
    "data/process/hasil_download_gabungan.csv",
    index=False
)

print("File berhasil disimpan!")

File berhasil disimpan!


In [ ]:
print("Daftar kategori Pintu Masuk:")

print(
    df_download_gabungan["Pintu Masuk"]
    .unique()
)

Daftar kategori Pintu Masuk:
['A. Pintu Udara' 'Adi Sucipto' 'Ahmad Yani' 'Aruk' 'Atambua'
 'B. Pintu Laut' 'Bandara Int. Lombok' 'Batam' 'C. Pintu Darat' 'Entikong'
 'Hasanuddin' 'Husein Sastranegara' 'Jayapura' 'Juanda' 'Jumlah (A+B+C)'
 'Kualanamu' 'Lainnya' 'Minangkabau' 'Nanga Badau' 'Ngurah Rai'
 'Pintu Darat Lainnya' 'Pintu Laut Lainnya' 'Pintu Udara Lainnya'
 'Sam Ratulangi' 'Soekarno-Hatta' 'Sultan Badaruddin II'
 'Sultan Iskandar Muda' 'Sultan Syarif Kasim II' 'Supadio'
 'Tanjung Balai Karimun' 'Tanjung Benoa' 'Tanjung Mas' 'Tanjung Pinang'
 'Tanjung Uban' 'Total']


In [ ]:
# Daftar baris yang bukan data pintu masuk individual

baris_dihapus = [
    "A. Pintu Udara",
    "B. Pintu Laut",
    "C. Pintu Darat",
    "Jumlah (A+B+C)",
    "Total"
]

# Menghapus baris kategori dan rekapitulasi

df_download_detail = df_download_gabungan[
    ~df_download_gabungan["Pintu Masuk"].isin(baris_dihapus)
].copy()

# Memeriksa hasil

print("Jumlah baris sebelum:", len(df_download_gabungan))
print("Jumlah baris sesudah:", len(df_download_detail))

print("\nDaftar pintu masuk:")
print(df_download_detail["Pintu Masuk"].unique())

Jumlah baris sebelum: 140
Jumlah baris sesudah: 120

Daftar pintu masuk:
['Adi Sucipto' 'Ahmad Yani' 'Aruk' 'Atambua' 'Bandara Int. Lombok' 'Batam'
 'Entikong' 'Hasanuddin' 'Husein Sastranegara' 'Jayapura' 'Juanda'
 'Kualanamu' 'Lainnya' 'Minangkabau' 'Nanga Badau' 'Ngurah Rai'
 'Pintu Darat Lainnya' 'Pintu Laut Lainnya' 'Pintu Udara Lainnya'
 'Sam Ratulangi' 'Soekarno-Hatta' 'Sultan Badaruddin II'
 'Sultan Iskandar Muda' 'Sultan Syarif Kasim II' 'Supadio'
 'Tanjung Balai Karimun' 'Tanjung Benoa' 'Tanjung Mas' 'Tanjung Pinang'
 'Tanjung Uban']


In [ ]:
# Memastikan folder penyimpanan tersedia
os.makedirs("data/process", exist_ok=True)

# Menyimpan dataset yang sudah dibersihkan
df_download_detail.to_csv(
    "data/process/hasil_download_bersih.csv",
    index=False
)

print("Dataset bersih berhasil disimpan!")
print("Jumlah baris:", len(df_download_detail))
print("Jumlah kolom:", len(df_download_detail.columns))

Dataset bersih berhasil disimpan!
Jumlah baris: 120
Jumlah kolom: 15


In [ ]:
# Pemeriksaan akhir dataset bersih

print("JUMLAH BARIS:", len(df_download_detail))
print("JUMLAH KOLOM:", len(df_download_detail.columns))

print("\nJUMLAH DATA KOSONG:")
print(df_download_detail.isnull().sum())

print("\nJUMLAH DUPLIKASI:")
print(df_download_detail.duplicated().sum())

print("\nDATA BERDASARKAN TAHUN:")
print(df_download_detail["Tahun"].value_counts().sort_index())

print("\nCONTOH DATA:")
display(df_download_detail.head(10))

JUMLAH BARIS: 120
JUMLAH KOLOM: 15

JUMLAH DATA KOSONG:
Pintu Masuk    0
Januari        5
Februari       5
Maret          5
April          5
Mei            5
Juni           5
Juli           5
Agustus        5
September      5
Oktober        5
November       5
Desember       4
Tahunan        4
Tahun          0
dtype: int64

JUMLAH DUPLIKASI:
0

DATA BERDASARKAN TAHUN:
Tahun
2022    30
2023    30
2024    30
2025    30
Name: count, dtype: int64

CONTOH DATA:


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022
10,Hasanuddin,0.0,0.0,0.0,164.0,364.0,523.0,706.0,503.0,701.0,897.0,526.0,1530.0,5914.0,2022
11,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0,2022
12,Jayapura,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,78.0,166.0,184.0,448.0,2022


In [ ]:
# Melihat baris yang memiliki data kosong

data_kosong = df_download_detail[
    df_download_detail.isnull().any(axis=1)
]

print("Jumlah baris dengan data kosong:",
      len(data_kosong))

display(data_kosong)

Jumlah baris dengan data kosong: 5


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
16,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
31,Tanjung Mas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
51,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
66,Tanjung Mas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
86,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2024


In [ ]:
# Daftar kolom bulan
kolom_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "November", "Desember"
]

# Menghapus baris yang seluruh data bulanannya kosong
df_download_final = df_download_detail.dropna(
    subset=kolom_bulan,
    how="all"
).copy()

# Memeriksa hasil
print("Jumlah baris sebelum:", len(df_download_detail))
print("Jumlah baris sesudah:", len(df_download_final))

print("\nData kosong tersisa:")
print(df_download_final.isnull().sum())

display(df_download_final.head(10))

Jumlah baris sebelum: 120
Jumlah baris sesudah: 116

Data kosong tersisa:
Pintu Masuk    0
Januari        1
Februari       1
Maret          1
April          1
Mei            1
Juni           1
Juli           1
Agustus        1
September      1
Oktober        1
November       1
Desember       0
Tahunan        0
Tahun          0
dtype: int64


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022
10,Hasanuddin,0.0,0.0,0.0,164.0,364.0,523.0,706.0,503.0,701.0,897.0,526.0,1530.0,5914.0,2022
11,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0,2022
12,Jayapura,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,78.0,166.0,184.0,448.0,2022


In [ ]:
df_download_final.to_csv(
    "data/process/hasil_download_final.csv",
    index=False
)

print("Dataset final berhasil disimpan!")

Dataset final berhasil disimpan!


In [ ]:
print("JUMLAH BARIS:", len(df_download_final))
print("JUMLAH KOLOM:", len(df_download_final.columns))

print("\nDATA BERDASARKAN TAHUN:")
print(
    df_download_final["Tahun"]
    .value_counts()
    .sort_index()
)

print("\nJUMLAH DATA KOSONG:")
print(df_download_final.isnull().sum())

print("\nJUMLAH DUPLIKASI:")
print(df_download_final.duplicated().sum())

JUMLAH BARIS: 116
JUMLAH KOLOM: 15

DATA BERDASARKAN TAHUN:
Tahun
2022    28
2023    28
2024    30
2025    30
Name: count, dtype: int64

JUMLAH DATA KOSONG:
Pintu Masuk    0
Januari        1
Februari       1
Maret          1
April          1
Mei            1
Juni           1
Juli           1
Agustus        1
September      1
Oktober        1
November       1
Desember       0
Tahunan        0
Tahun          0
dtype: int64

JUMLAH DUPLIKASI:
0


In [ ]:
import hashlib
import os

# Lokasi file dataset asli
file_asli = (
    "data/raw/download/"
    "Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia "
    "Menurut Pintu Masuk, 2008 - sekarang, 2022.csv"
)

# Menghitung MD5
def hitung_md5(lokasi):
    md5 = hashlib.md5()

    with open(lokasi, "rb") as file:
        for bagian in iter(lambda: file.read(4096), b""):
            md5.update(bagian)

    return md5.hexdigest()

print("MD5:", hitung_md5(file_asli))
print("Ukuran file:", os.path.getsize(file_asli), "byte")

MD5: daba6b473fdf84e7d075cc32e7c22122
Ukuran file: 2706 byte


In [ ]:
import hashlib
import os
import pandas as pd

# Fungsi menghitung MD5
def hitung_md5(lokasi):
    md5 = hashlib.md5()

    with open(lokasi, "rb") as file:
        for bagian in iter(lambda: file.read(4096), b""):
            md5.update(bagian)

    return md5.hexdigest()


# Menghitung ukuran dan MD5 semua dataset
daftar_metadata = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    metadata = {
        "nama_file": file,
        "ukuran_byte": os.path.getsize(lokasi),
        "md5": hitung_md5(lokasi)
    }

    daftar_metadata.append(metadata)


# Menampilkan hasil
df_metadata = pd.DataFrame(daftar_metadata)

display(df_metadata)

,nama_file,ukuran_byte,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,2706,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3065,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3112,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3188,0a25d32a1fc15e65974727e20f22a783


In [ ]:
# Membuat data provenance

from datetime import date

daftar_provenance = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    daftar_provenance.append({
        "nama_sumber": file,
        "url": "ISI_URL_SUMBER_ASLI",
        "tanggal_akses": "2026-11-24",
        "lisensi": "PERLU_VERIFIKASI",
        "md5": hitung_md5(lokasi)
    })

df_provenance = pd.DataFrame(daftar_provenance)

# Menyimpan provenance
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,0a25d32a1fc15e65974727e20f22a783


In [ ]:
# Membuat provenance untuk 4 dataset BPS

daftar_provenance = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    daftar_provenance.append({
        "nama_sumber": file,
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": hitung_md5(lokasi)
    })

df_provenance = pd.DataFrame(daftar_provenance)

# Menyimpan provenance
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783


In [ ]:
df_provenance = pd.read_csv("docs/provenance.csv")

print("Jumlah sumber:", len(df_provenance))

display(df_provenance)

In [ ]:
# Mencari file HTML di folder scraping

folder_scraping = "data/raw/scraping"

for file in os.listdir(folder_scraping):
    print(file)

day_bali_tour.html


In [ ]:
# Menghitung metadata file HTML

for file in os.listdir(folder_scraping):

    lokasi = os.path.join(folder_scraping, file)

    if file.endswith(".html"):
        print("Nama file:", file)
        print("Ukuran:", os.path.getsize(lokasi), "byte")
        print("MD5:", hitung_md5(lokasi))

Nama file: day_bali_tour.html
Ukuran: 391767 byte
MD5: b27ca0b9e23f86169d62b0a869ab37bd


In [ ]:
# Membaca provenance yang sudah ada
df_provenance = pd.read_csv("docs/provenance.csv")

# Membuat data sumber scraping
data_scraping = {
    "nama_sumber": "day_bali_tour.html",
    "url": "https://www.daybalitour.com/best-bali-day-tour-packages/",
    "tanggal_akses": "2026-11-24",
    "lisensi": "Belum dinyatakan pada halaman sumber",
    "md5": "a2d4750647a51c5252cb8b8bf0d28996"
}

# Menambahkan sumber scraping ke metadata
df_provenance = pd.concat(
    [df_provenance, pd.DataFrame([data_scraping])],
    ignore_index=True
)

# Menyimpan kembali provenance
df_provenance.to_csv("docs/provenance.csv", index=False)

# Menampilkan hasil
display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783
4,day_bali_tour.html,https://www.daybalitour.com/best-bali-day-tour...,2026-11-24,Belum dinyatakan pada halaman sumber,a2d4750647a51c5252cb8b8bf0d28996


In [ ]:
# Melihat hasil scraping
display(df_scraping.head())

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...


In [ ]:
# 1. Menghapus baris tanpa harga
df_scraping_bersih = df_scraping.dropna(
    subset=["harga"]
).copy()

# 2. Menghapus baris yang bukan nama paket wisata
df_scraping_bersih = df_scraping_bersih[
    ~df_scraping_bersih["nama_paket"].str.startswith(
        ("From", "IDR"), na=False
    )
].copy()

# 3. Menghapus data duplikat
df_scraping_bersih = df_scraping_bersih.drop_duplicates()

# 4. Mengatur ulang indeks
df_scraping_bersih = df_scraping_bersih.reset_index(drop=True)

# 5. Menampilkan hasil
display(df_scraping_bersih)

# 6. Menghitung jumlah data
print("Jumlah data:", len(df_scraping_bersih))

,nama_paket,harga,informasi
0,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
1,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
2,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
3,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
4,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...
5,Bali Kintamani Volcano View Tour Package,IDR 715.000,Bali Kintamani Volcano View Tour Package From ...
6,Bali Driver with Toyota Innova,IDR 1.125.000,Bali Driver with Toyota Innova IDR 1.125.000/C...
7,Bali Driver with Toyota Hiace,IDR 1.275.000,Bali Driver with Toyota Hiace IDR 1.275.000/Ca...


Jumlah data: 8


In [ ]:
# Menyimpan hasil scraping yang sudah dibersihkan
df_scraping_bersih.to_csv(
    "data/process/hasil_scraping_daybalitour.csv",
    index=False
)

print("Data scraping berhasil disimpan!")

Data scraping berhasil disimpan!


In [ ]:
import os

file_scraping = "data/process/hasil_scraping_daybalitour.csv"

print("File tersedia:", os.path.exists(file_scraping))
print("Ukuran file:", os.path.getsize(file_scraping), "byte")

File tersedia: True
Ukuran file: 2217 byte


In [ ]:
# Memeriksa informasi dataset
print("Informasi dataset:")
df_scraping_bersih.info()

# Memeriksa nilai kosong
print("\nJumlah nilai kosong:")
print(df_scraping_bersih.isnull().sum())

# Memeriksa data duplikat
print("\nJumlah baris duplikat:")
print(df_scraping_bersih.duplicated().sum())

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   nama_paket  8 non-null      object
 1   harga       8 non-null      object
 2   informasi   8 non-null      object
dtypes: object(3)
memory usage: 324.0+ bytes

Jumlah nilai kosong:
nama_paket    0
harga         0
informasi     0
dtype: int64

Jumlah baris duplikat:
0


In [ ]:
# Menampilkan nama paket dan harga
display(
    df_scraping_bersih[["nama_paket", "harga"]]
)

,nama_paket,harga
0,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000
1,Bali Best Nusa Penida Island Tour Packages,IDR 760.000
2,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000
3,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000
4,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000
5,Bali Kintamani Volcano View Tour Package,IDR 715.000
6,Bali Driver with Toyota Innova,IDR 1.125.000
7,Bali Driver with Toyota Hiace,IDR 1.275.000


In [ ]:
# Menghapus tulisan IDR dan titik pemisah ribuan
df_scraping_bersih["harga"] = (
    df_scraping_bersih["harga"]
    .str.replace("IDR", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.strip()
    .astype(int)
)

# Memeriksa hasil konversi
display(df_scraping_bersih[["nama_paket", "harga"]])

# Memeriksa tipe data
print(df_scraping_bersih.dtypes)

,nama_paket,harga
0,Bali Waterfall and Best Ubud Tour Packages,580000
1,Bali Best Nusa Penida Island Tour Packages,760000
2,Bali Lempuyang Gate of Heaven Tour Package,745000
3,Bali Ulun Danu and Tanah Lot Tour Package,790000
4,Bali Tanjung Benoa & Uluwatu Tour Package,870000
5,Bali Kintamani Volcano View Tour Package,715000
6,Bali Driver with Toyota Innova,1125000
7,Bali Driver with Toyota Hiace,1275000


nama_paket    object
harga          int64
informasi     object
dtype: object


In [ ]:
# Menghitung statistik harga
print("Harga termurah: Rp", df_scraping_bersih["harga"].min())
print("Harga termahal: Rp", df_scraping_bersih["harga"].max())
print("Rata-rata harga: Rp", round(df_scraping_bersih["harga"].mean()))

# Statistik lengkap
display(df_scraping_bersih["harga"].describe())

Harga termurah: Rp 580000
Harga termahal: Rp 1275000
Rata-rata harga: Rp 857500


,harga
count,8.000000e+00
mean,8.575000e+05
std,2.299223e+05
min,5.800000e+05
25%,7.375000e+05
50%,7.750000e+05
75%,9.337500e+05
max,1.275000e+06


In [ ]:
df_scraping_bersih.to_csv(
    "data/process/hasil_scraping_daybalitour.csv",
    index=False
)

print("Dataset terbaru berhasil disimpan!")

Dataset terbaru berhasil disimpan!


In [ ]:
import pandas as pd

checklist = pd.DataFrame({
    "Aspek": [
        "Sumber data dan URL dicantumkan",
        "Lisensi data diperiksa dan dicatat",
        "robots.txt / Terms of Service diperiksa",
        "Kolom PII diidentifikasi",
        "PII dihapus, di-mask, atau di-pseudonimisasi",
        "Prinsip minimalisasi data diterapkan",
        "Potensi bias sampel dianalisis",
        "Tujuan penggunaan data dinyatakan eksplisit",
        "Rencana retensi dan penghapusan data ada"
    ],

    "Status (Ya/Tidak/NA)": [
        "Ya",
        "Tidak",
        "Tidak",
        "Ya",
        "NA",
        "Ya",
        "Ya",
        "Ya",
        "Ya"
    ],

    "Bukti / Catatan": [
        "URL Day Bali Tour dan BPS dicatat dalam provenance.csv.",
        "Lisensi penggunaan ulang belum dapat dipastikan.",
        "robots.txt diperiksa; Terms of Service belum diperiksa.",
        "Tidak ditemukan PII pada kolom dataset yang diperiksa.",
        "Tidak diperlukan karena tidak ditemukan PII.",
        "Hanya mengambil nama paket, harga, dan informasi layanan.",
        "Sampel hanya berasal dari satu situs sehingga tidak mewakili seluruh penyedia wisata Bali.",
        "Data digunakan untuk tugas UTS Akuisisi dan Manajemen Data.",
        "Data disimpan selama pengerjaan dan penilaian UTS, kemudian dihapus jika tidak lagi diperlukan."
    ]
})

# Menyimpan checklist
checklist.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False
)

display(checklist)

,Aspek,Status (Ya/Tidak/NA),Bukti / Catatan
0,Sumber data dan URL dicantumkan,Ya,URL Day Bali Tour dan BPS dicatat dalam proven...
1,Lisensi data diperiksa dan dicatat,Tidak,Lisensi penggunaan ulang belum dapat dipastikan.
2,robots.txt / Terms of Service diperiksa,Tidak,robots.txt diperiksa; Terms of Service belum d...
3,Kolom PII diidentifikasi,Ya,Tidak ditemukan PII pada kolom dataset yang di...
4,"PII dihapus, di-mask, atau di-pseudonimisasi",NA,Tidak diperlukan karena tidak ditemukan PII.
5,Prinsip minimalisasi data diterapkan,Ya,"Hanya mengambil nama paket, harga, dan informa..."
6,Potensi bias sampel dianalisis,Ya,Sampel hanya berasal dari satu situs sehingga ...
7,Tujuan penggunaan data dinyatakan eksplisit,Ya,Data digunakan untuk tugas UTS Akuisisi dan Ma...
8,Rencana retensi dan penghapusan data ada,Ya,Data disimpan selama pengerjaan dan penilaian ...


In [ ]:
import pandas as pd
import os

# Memastikan folder docs tersedia
os.makedirs("docs", exist_ok=True)

# Membuat checklist etika data
checklist = pd.DataFrame({
    "Aspek": [
        "Sumber data dan URL dicantumkan",
        "Lisensi data diperiksa dan dicatat",
        "robots.txt / Terms of Service diperiksa",
        "Kolom PII diidentifikasi",
        "PII dihapus, di-mask, atau di-pseudonimisasi",
        "Prinsip minimalisasi data diterapkan",
        "Potensi bias sampel dianalisis",
        "Tujuan penggunaan data dinyatakan eksplisit",
        "Rencana retensi dan penghapusan data ada"
    ],

    "Status": [
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "NA",
        "Ya",
        "Ya",
        "Ya",
        "Ya"
    ],

    "Bukti / Catatan": [
        "URL sumber BPS dan Day Bali Tour dicatat dalam docs/provenance.csv.",
        "Lisensi diperiksa dan dicatat. Lisensi penggunaan ulang Day Bali Tour belum ditemukan sehingga izinnya belum dapat dipastikan.",
        "robots.txt diperiksa dan halaman paket tidak diblokir. Terms of Service tidak ditemukan pada website.",
        "Kolom dataset diperiksa. Tidak ditemukan data pribadi seperti nama individu, nomor telepon, alamat pribadi, atau email pribadi.",
        "Tidak diperlukan karena pada kolom yang diperiksa tidak ditemukan PII.",
        "Data yang diambil dibatasi pada data kunjungan wisatawan serta nama paket, harga, dan informasi layanan.",
        "Data paket wisata hanya berasal dari satu website sehingga belum mewakili seluruh penyedia wisata di Bali.",
        "Data digunakan untuk memenuhi tugas UTS Akuisisi dan Manajemen Data.",
        "Data disimpan selama pengerjaan dan penilaian UTS. Setelah tidak diperlukan lagi, data akan dihapus dari penyimpanan kerja."
    ]
})

# Menyimpan checklist sebagai CSV
checklist.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False
)

# Menampilkan hasil
display(checklist)

print("Checklist berhasil disimpan!")

,Aspek,Status,Bukti / Catatan
0,Sumber data dan URL dicantumkan,Ya,URL sumber BPS dan Day Bali Tour dicatat dalam...
1,Lisensi data diperiksa dan dicatat,Ya,Lisensi diperiksa dan dicatat. Lisensi penggun...
2,robots.txt / Terms of Service diperiksa,Ya,robots.txt diperiksa dan halaman paket tidak d...
3,Kolom PII diidentifikasi,Ya,Kolom dataset diperiksa. Tidak ditemukan data ...
4,"PII dihapus, di-mask, atau di-pseudonimisasi",NA,Tidak diperlukan karena pada kolom yang diperi...
5,Prinsip minimalisasi data diterapkan,Ya,Data yang diambil dibatasi pada data kunjungan...
6,Potensi bias sampel dianalisis,Ya,Data paket wisata hanya berasal dari satu webs...
7,Tujuan penggunaan data dinyatakan eksplisit,Ya,Data digunakan untuk memenuhi tugas UTS Akuisi...
8,Rencana retensi dan penghapusan data ada,Ya,Data disimpan selama pengerjaan dan penilaian ...


Checklist berhasil disimpan!


In [ ]:
import os

file_checklist = "docs/data_ethics_checklist.csv"

print("File tersedia:", os.path.exists(file_checklist))

if os.path.exists(file_checklist):
    print("Ukuran file:", os.path.getsize(file_checklist), "byte")
    print("\nIsi checklist:")
    display(pd.read_csv(file_checklist))

File tersedia: True
Ukuran file: 1295 byte

Isi checklist:


,Aspek,Status,Bukti / Catatan
0,Sumber data dan URL dicantumkan,Ya,URL sumber BPS dan Day Bali Tour dicatat dalam...
1,Lisensi data diperiksa dan dicatat,Ya,Lisensi diperiksa dan dicatat. Lisensi penggun...
2,robots.txt / Terms of Service diperiksa,Ya,robots.txt diperiksa dan halaman paket tidak d...
3,Kolom PII diidentifikasi,Ya,Kolom dataset diperiksa. Tidak ditemukan data ...
4,"PII dihapus, di-mask, atau di-pseudonimisasi",NaN,Tidak diperlukan karena pada kolom yang diperi...
5,Prinsip minimalisasi data diterapkan,Ya,Data yang diambil dibatasi pada data kunjungan...
6,Potensi bias sampel dianalisis,Ya,Data paket wisata hanya berasal dari satu webs...
7,Tujuan penggunaan data dinyatakan eksplisit,Ya,Data digunakan untuk memenuhi tugas UTS Akuisi...
8,Rencana retensi dan penghapusan data ada,Ya,Data disimpan selama pengerjaan dan penilaian ...


In [ ]:
import requests
import os

url_robots = "https://www.daybalitour.com/robots.txt"

headers = {
    "User-Agent": "UTS-Data-Acquisition (email: ahmadpaesoel34@gmail.com)"
}

response = requests.get(
    url_robots,
    headers=headers,
    timeout=15
)

os.makedirs("docs/evidence", exist_ok=True)

with open(
    "docs/evidence/robots_daybalitour.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(response.text)

print("Status HTTP:", response.status_code)
print("Bukti robots.txt berhasil disimpan!")

Status HTTP: 200
Bukti robots.txt berhasil disimpan!


In [ ]:
catatan_terms = """
HASIL PEMERIKSAAN TERMS OF SERVICE

Nama website: Day Bali Tour
URL:
https://www.daybalitour.com/best-bali-day-tour-packages/

Hasil pemeriksaan:
Tidak ditemukan halaman Terms of Service atau Terms and Conditions
pada website setelah pemeriksaan bagian bawah halaman.

Pemeriksaan robots.txt:
Berhasil diakses dengan status HTTP 200.
Halaman paket wisata yang digunakan tidak tercantum dalam aturan Disallow.

Kesimpulan:
Izin penggunaan ulang konten belum dapat dipastikan.
Robots.txt bukan merupakan izin penggunaan ulang konten.
"""

with open(
    "docs/evidence/catatan_terms_daybalitour.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(catatan_terms)

print("Catatan pemeriksaan berhasil disimpan!")

Catatan pemeriksaan berhasil disimpan!


In [ ]:
import os

for nama_file in os.listdir("docs/evidence"):
    print(nama_file)

robots_daybalitour.txt
catatan_terms_daybalitour.txt


In [ ]:
print("=== DATA DOWNLOAD BPS ===")
print("Jumlah baris:", len(df_download_final))
print("Jumlah kolom:", len(df_download_final.columns))
print("Duplikat:", df_download_final.duplicated().sum())

print("\n=== DATA SCRAPING DAY BALI TOUR ===")
print("Jumlah baris:", len(df_scraping_bersih))
print("Jumlah kolom:", len(df_scraping_bersih.columns))
print("Duplikat:", df_scraping_bersih.duplicated().sum())

print("\n=== KOLOM DATA DOWNLOAD ===")
print(df_download_final.columns.tolist())

print("\n=== KOLOM DATA SCRAPING ===")
print(df_scraping_bersih.columns.tolist())

=== DATA DOWNLOAD BPS ===
Jumlah baris: 116
Jumlah kolom: 15
Duplikat: 0

=== DATA SCRAPING DAY BALI TOUR ===
Jumlah baris: 8
Jumlah kolom: 3
Duplikat: 0

=== KOLOM DATA DOWNLOAD ===
['Pintu Masuk', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember', 'Tahunan', 'Tahun']

=== KOLOM DATA SCRAPING ===
['nama_paket', 'harga', 'informasi']


In [ ]:
# Melihat data BPS yang berkaitan dengan Bali
print("DATA BPS:")
display(
    df_download_final[
        df_download_final["Pintu Masuk"].str.contains(
            "Ngurah Rai", case=False, na=False
        )
    ][["Pintu Masuk", "Tahun", "Tahunan"]]
)

print("\nDATA DAY BALI TOUR:")
display(df_scraping_bersih)

DATA BPS:


,Pintu Masuk,Tahun,Tahunan
19,Ngurah Rai,2022,2154045.0
54,Ngurah Rai,2023,5248113.0
89,Ngurah Rai,2024,6308541.0
124,Ngurah Rai,2025,6907585.0



DATA DAY BALI TOUR:


,nama_paket,harga,informasi
0,Bali Waterfall and Best Ubud Tour Packages,580000,Bali Waterfall and Best Ubud Tour Packages Fro...
1,Bali Best Nusa Penida Island Tour Packages,760000,Bali Best Nusa Penida Island Tour Packages Fro...
2,Bali Lempuyang Gate of Heaven Tour Package,745000,Bali Lempuyang Gate of Heaven Tour Package Fro...
3,Bali Ulun Danu and Tanah Lot Tour Package,790000,Bali Ulun Danu and Tanah Lot Tour Package From...
4,Bali Tanjung Benoa & Uluwatu Tour Package,870000,Bali Tanjung Benoa & Uluwatu Tour Package From...
5,Bali Kintamani Volcano View Tour Package,715000,Bali Kintamani Volcano View Tour Package From ...
6,Bali Driver with Toyota Innova,1125000,Bali Driver with Toyota Innova IDR 1.125.000/C...
7,Bali Driver with Toyota Hiace,1275000,Bali Driver with Toyota Hiace IDR 1.275.000/Ca...


In [ ]:
# Melihat seluruh informasi paket wisata
for index, baris in df_scraping_bersih.iterrows():
    print(f"\nPaket {index + 1}")
    print("Nama:", baris["nama_paket"])
    print("Informasi:", baris["informasi"])


Paket 1
Nama: Bali Waterfall and Best Ubud Tour Packages
Informasi: Bali Waterfall and Best Ubud Tour Packages From IDR 580.000 What’s Included: Private Hotel Pick-up and Drop-off Private Car with Air Conditioning Driver (English speaking) Lunch (Indonesian Food) Bottled Water All Entrance Fees

Paket 2
Nama: Bali Best Nusa Penida Island Tour Packages
Informasi: Bali Best Nusa Penida Island Tour Packages From IDR 760.000 What’s Included: Private Hotel Pick-up and Drop-off Private Car with Air Conditioning Driver (English speaking) Lunch (Indonesian Food) Bottled Water All Entrance Fees

Paket 3
Nama: Bali Lempuyang Gate of Heaven Tour Package
Informasi: Bali Lempuyang Gate of Heaven Tour Package From IDR 745.000 What’s Included: Private Hotel Pick-up and Drop-off Private Car with Air Conditioning Driver (English speaking) Lunch (Indonesian Food) Bottled Water All Entrance Fees

Paket 4
Nama: Bali Ulun Danu and Tanah Lot Tour Package
Informasi: Bali Ulun Danu and Tanah Lot Tour Package

In [ ]:
import requests
import time

# Alamat API Nominatim
url = "https://nominatim.openstreetmap.org/search"

# Parameter pencarian lokasi
params = {
    "q": "Ubud, Bali, Indonesia",
    "format": "jsonv2",
    "addressdetails": 1,
    "limit": 1
}

# Identitas aplikasi
headers = {
    "User-Agent": "UTS-Data-Acquisition (email: EMAIL_KAMPUS_KAMU)"
}

# Mengirim permintaan API
response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=20
)

print("Status HTTP:", response.status_code)

# Memeriksa hasil
if response.status_code == 200:
    hasil_api = response.json()

    if len(hasil_api) > 0:
        print("Nama lokasi:", hasil_api[0]["display_name"])
        print("Latitude:", hasil_api[0]["lat"])
        print("Longitude:", hasil_api[0]["lon"])
        print("Alamat:", hasil_api[0].get("address", {}))
    else:
        print("Lokasi tidak ditemukan.")
else:
    print("Permintaan API gagal:", response.text)

Status HTTP: 200
Nama lokasi: Ubud, Gianyar, Bali, Indonesia
Latitude: -8.5170195
Longitude: 115.2550507
Alamat: {'town': 'Ubud', 'region': 'Gianyar', 'state': 'Bali', 'ISO3166-2-lvl4': 'ID-BA', 'country': 'Indonesia', 'country_code': 'id'}


In [ ]:
import requests
import time
import pandas as pd
import os
import json

# Daftar destinasi
daftar_destinasi = [
    "Nusa Penida, Bali, Indonesia",
    "Lempuyang Temple, Bali, Indonesia",
    "Ulun Danu Beratan, Bali, Indonesia",
    "Tanah Lot, Bali, Indonesia",
    "Tanjung Benoa, Bali, Indonesia",
    "Uluwatu Temple, Bali, Indonesia",
    "Kintamani, Bali, Indonesia"
]

url = "https://nominatim.openstreetmap.org/search"

headers = {
    "User-Agent": "UTS-Data-Acquisition (email: EMAIL_KAMPUS_KAMU)"
}

hasil_semua = []

for destinasi in daftar_destinasi:

    params = {
        "q": destinasi,
        "format": "jsonv2",
        "addressdetails": 1,
        "limit": 1
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=20
        )

        response.raise_for_status()
        hasil = response.json()

        if hasil:
            lokasi = hasil[0]

            hasil_semua.append({
                "destinasi": destinasi,
                "nama_lokasi": lokasi.get("display_name"),
                "latitude": lokasi.get("lat"),
                "longitude": lokasi.get("lon"),
                "alamat": lokasi.get("address", {})
            })

            print("Ditemukan:", destinasi)

        else:
            print("Tidak ditemukan:", destinasi)

    except requests.RequestException as error:
        print("Gagal:", destinasi, error)

    # Jeda agar mematuhi batas penggunaan API
    time.sleep(1.1)

# Membuat DataFrame
df_api = pd.DataFrame(hasil_semua)

display(df_api)
print("Jumlah lokasi ditemukan:", len(df_api))

Ditemukan: Nusa Penida, Bali, Indonesia
Ditemukan: Lempuyang Temple, Bali, Indonesia
Ditemukan: Ulun Danu Beratan, Bali, Indonesia
Ditemukan: Tanah Lot, Bali, Indonesia
Ditemukan: Tanjung Benoa, Bali, Indonesia
Ditemukan: Uluwatu Temple, Bali, Indonesia
Ditemukan: Kintamani, Bali, Indonesia


,destinasi,nama_lokasi,latitude,longitude,alamat
0,"Nusa Penida, Bali, Indonesia","Nusa Penida, Klungkung, Bali, Indonesia",-8.7455877,115.5346805,"{'island': 'Nusa Penida', 'town': 'Nusa Penida..."
1,"Lempuyang Temple, Bali, Indonesia","Pura Lempuyang Luhur, Temple Path Steps, Tri B...",-8.3951892,115.6480606,"{'amenity': 'Pura Lempuyang Luhur', 'road': 'T..."
2,"Ulun Danu Beratan, Bali, Indonesia","Pura Ulun Danu Beratan, 2, Kembangkerta, Candi...",-8.2755069,115.1662600,"{'amenity': 'Pura Ulun Danu Beratan', 'road': ..."
3,"Tanah Lot, Bali, Indonesia","Tanah Lot, Jalan Enjung Pura, Batugaing, Berab...",-8.6212286,115.0868849,"{'amenity': 'Tanah Lot', 'road': 'Jalan Enjung..."
4,"Tanjung Benoa, Bali, Indonesia","Tanjung Benoa, Gang Guntur, Benoa, Kuta Selata...",-8.7657398,115.2232562,"{'natural': 'Tanjung Benoa', 'road': 'Gang Gun..."
5,"Uluwatu Temple, Bali, Indonesia","Pura Luhur Uluwatu, Jalan Raya Uluwatu, Pecatu...",-8.8293693,115.0843428,"{'amenity': 'Pura Luhur Uluwatu', 'road': 'Jal..."
6,"Kintamani, Bali, Indonesia","Kintamani, Bangli, Bali, Indonesia",-8.2495163,115.3482233,"{'town': 'Kintamani', 'region': 'Bangli', 'sta..."


Jumlah lokasi ditemukan: 7


In [ ]:
# Menambahkan hasil pengujian Ubud ke dataset API
data_ubud = pd.DataFrame([{
    "destinasi": "Ubud, Bali, Indonesia",
    "nama_lokasi": "Ubud, Gianyar, Bali, Indonesia",
    "latitude": "-8.5170195",
    "longitude": "115.2550507",
    "alamat": {
        "town": "Ubud",
        "region": "Gianyar",
        "state": "Bali",
        "country": "Indonesia"
    }
}])

df_api = pd.concat(
    [df_api, data_ubud],
    ignore_index=True
)

print("Jumlah data API:", len(df_api))
display(df_api[[
    "destinasi", "nama_lokasi", "latitude", "longitude"
]])

Jumlah data API: 8


,destinasi,nama_lokasi,latitude,longitude
0,"Nusa Penida, Bali, Indonesia","Nusa Penida, Klungkung, Bali, Indonesia",-8.7455877,115.5346805
1,"Lempuyang Temple, Bali, Indonesia","Pura Lempuyang Luhur, Temple Path Steps, Tri B...",-8.3951892,115.6480606
2,"Ulun Danu Beratan, Bali, Indonesia","Pura Ulun Danu Beratan, 2, Kembangkerta, Candi...",-8.2755069,115.1662600
3,"Tanah Lot, Bali, Indonesia","Tanah Lot, Jalan Enjung Pura, Batugaing, Berab...",-8.6212286,115.0868849
4,"Tanjung Benoa, Bali, Indonesia","Tanjung Benoa, Gang Guntur, Benoa, Kuta Selata...",-8.7657398,115.2232562
5,"Uluwatu Temple, Bali, Indonesia","Pura Luhur Uluwatu, Jalan Raya Uluwatu, Pecatu...",-8.8293693,115.0843428
6,"Kintamani, Bali, Indonesia","Kintamani, Bangli, Bali, Indonesia",-8.2495163,115.3482233
7,"Ubud, Bali, Indonesia","Ubud, Gianyar, Bali, Indonesia",-8.5170195,115.2550507


In [ ]:
# Menyamakan nama destinasi pada data API
pemetaan_api = {
    "Nusa Penida, Bali, Indonesia": "Nusa Penida",
    "Lempuyang Temple, Bali, Indonesia": "Lempuyang",
    "Ulun Danu Beratan, Bali, Indonesia": "Ulun Danu Beratan",
    "Tanah Lot, Bali, Indonesia": "Tanah Lot",
    "Tanjung Benoa, Bali, Indonesia": "Tanjung Benoa",
    "Uluwatu Temple, Bali, Indonesia": "Uluwatu",
    "Kintamani, Bali, Indonesia": "Kintamani",
    "Ubud, Bali, Indonesia": "Ubud"
}

df_api["destinasi_key"] = df_api["destinasi"].map(pemetaan_api)

display(df_api[[
    "destinasi_key", "nama_lokasi", "latitude", "longitude"
]])

,destinasi_key,nama_lokasi,latitude,longitude
0,Nusa Penida,"Nusa Penida, Klungkung, Bali, Indonesia",-8.7455877,115.5346805
1,Lempuyang,"Pura Lempuyang Luhur, Temple Path Steps, Tri B...",-8.3951892,115.6480606
2,Ulun Danu Beratan,"Pura Ulun Danu Beratan, 2, Kembangkerta, Candi...",-8.2755069,115.1662600
3,Tanah Lot,"Tanah Lot, Jalan Enjung Pura, Batugaing, Berab...",-8.6212286,115.0868849
4,Tanjung Benoa,"Tanjung Benoa, Gang Guntur, Benoa, Kuta Selata...",-8.7657398,115.2232562
5,Uluwatu,"Pura Luhur Uluwatu, Jalan Raya Uluwatu, Pecatu...",-8.8293693,115.0843428
6,Kintamani,"Kintamani, Bangli, Bali, Indonesia",-8.2495163,115.3482233
7,Ubud,"Ubud, Gianyar, Bali, Indonesia",-8.5170195,115.2550507


In [ ]:
data_penghubung = [
    {
        "nama_paket": "Bali Waterfall and Best Ubud Tour Packages",
        "destinasi_key": "Ubud"
    },
    {
        "nama_paket": "Bali Best Nusa Penida Island Tour Packages",
        "destinasi_key": "Nusa Penida"
    },
    {
        "nama_paket": "Bali Lempuyang Gate of Heaven Tour Package",
        "destinasi_key": "Lempuyang"
    },
    {
        "nama_paket": "Bali Ulun Danu and Tanah Lot Tour Package",
        "destinasi_key": "Ulun Danu Beratan"
    },
    {
        "nama_paket": "Bali Ulun Danu and Tanah Lot Tour Package",
        "destinasi_key": "Tanah Lot"
    },
    {
        "nama_paket": "Bali Tanjung Benoa & Uluwatu Tour Package",
        "destinasi_key": "Tanjung Benoa"
    },
    {
        "nama_paket": "Bali Tanjung Benoa & Uluwatu Tour Package",
        "destinasi_key": "Uluwatu"
    },
    {
        "nama_paket": "Bali Kintamani Volcano View Tour Package",
        "destinasi_key": "Kintamani"
    }
]

df_penghubung = pd.DataFrame(data_penghubung)

display(df_penghubung)

,nama_paket,destinasi_key
0,Bali Waterfall and Best Ubud Tour Packages,Ubud
1,Bali Best Nusa Penida Island Tour Packages,Nusa Penida
2,Bali Lempuyang Gate of Heaven Tour Package,Lempuyang
3,Bali Ulun Danu and Tanah Lot Tour Package,Ulun Danu Beratan
4,Bali Ulun Danu and Tanah Lot Tour Package,Tanah Lot
5,Bali Tanjung Benoa & Uluwatu Tour Package,Tanjung Benoa
6,Bali Tanjung Benoa & Uluwatu Tour Package,Uluwatu
7,Bali Kintamani Volcano View Tour Package,Kintamani


In [ ]:
df_scraping_lokasi = df_penghubung.merge(
    df_scraping_bersih,
    on="nama_paket",
    how="left",
    validate="many_to_one",
    indicator=True
)

display(df_scraping_lokasi)

,nama_paket,destinasi_key,harga,informasi,_merge
0,Bali Waterfall and Best Ubud Tour Packages,Ubud,580000,Bali Waterfall and Best Ubud Tour Packages Fro...,both
1,Bali Best Nusa Penida Island Tour Packages,Nusa Penida,760000,Bali Best Nusa Penida Island Tour Packages Fro...,both
2,Bali Lempuyang Gate of Heaven Tour Package,Lempuyang,745000,Bali Lempuyang Gate of Heaven Tour Package Fro...,both
3,Bali Ulun Danu and Tanah Lot Tour Package,Ulun Danu Beratan,790000,Bali Ulun Danu and Tanah Lot Tour Package From...,both
4,Bali Ulun Danu and Tanah Lot Tour Package,Tanah Lot,790000,Bali Ulun Danu and Tanah Lot Tour Package From...,both
5,Bali Tanjung Benoa & Uluwatu Tour Package,Tanjung Benoa,870000,Bali Tanjung Benoa & Uluwatu Tour Package From...,both
6,Bali Tanjung Benoa & Uluwatu Tour Package,Uluwatu,870000,Bali Tanjung Benoa & Uluwatu Tour Package From...,both
7,Bali Kintamani Volcano View Tour Package,Kintamani,715000,Bali Kintamani Volcano View Tour Package From ...,both


In [ ]:
print("Jumlah baris:", len(df_scraping_lokasi))

print("\nStatus penggabungan:")
print(df_scraping_lokasi["_merge"].value_counts())

print("\nData yang tidak cocok:")
display(
    df_scraping_lokasi[
        df_scraping_lokasi["_merge"] != "both"
    ]
)

Jumlah baris: 8

Status penggabungan:
_merge
both          8
left_only     0
right_only    0
Name: count, dtype: int64

Data yang tidak cocok:


,nama_paket,destinasi_key,harga,informasi,_merge


In [ ]:
df_hasil_merge = df_scraping_lokasi.merge(
    df_api,
    on="destinasi_key",
    how="left",
    validate="many_to_one",
    indicator="status_api"
)

display(df_hasil_merge)

,nama_paket,destinasi_key,harga,informasi,_merge,destinasi,nama_lokasi,latitude,longitude,alamat,status_api
0,Bali Waterfall and Best Ubud Tour Packages,Ubud,580000,Bali Waterfall and Best Ubud Tour Packages Fro...,both,"Ubud, Bali, Indonesia","Ubud, Gianyar, Bali, Indonesia",-8.5170195,115.2550507,"{'town': 'Ubud', 'region': 'Gianyar', 'state':...",both
1,Bali Best Nusa Penida Island Tour Packages,Nusa Penida,760000,Bali Best Nusa Penida Island Tour Packages Fro...,both,"Nusa Penida, Bali, Indonesia","Nusa Penida, Klungkung, Bali, Indonesia",-8.7455877,115.5346805,"{'island': 'Nusa Penida', 'town': 'Nusa Penida...",both
2,Bali Lempuyang Gate of Heaven Tour Package,Lempuyang,745000,Bali Lempuyang Gate of Heaven Tour Package Fro...,both,"Lempuyang Temple, Bali, Indonesia","Pura Lempuyang Luhur, Temple Path Steps, Tri B...",-8.3951892,115.6480606,"{'amenity': 'Pura Lempuyang Luhur', 'road': 'T...",both
3,Bali Ulun Danu and Tanah Lot Tour Package,Ulun Danu Beratan,790000,Bali Ulun Danu and Tanah Lot Tour Package From...,both,"Ulun Danu Beratan, Bali, Indonesia","Pura Ulun Danu Beratan, 2, Kembangkerta, Candi...",-8.2755069,115.1662600,"{'amenity': 'Pura Ulun Danu Beratan', 'road': ...",both
4,Bali Ulun Danu and Tanah Lot Tour Package,Tanah Lot,790000,Bali Ulun Danu and Tanah Lot Tour Package From...,both,"Tanah Lot, Bali, Indonesia","Tanah Lot, Jalan Enjung Pura, Batugaing, Berab...",-8.6212286,115.0868849,"{'amenity': 'Tanah Lot', 'road': 'Jalan Enjung...",both
5,Bali Tanjung Benoa & Uluwatu Tour Package,Tanjung Benoa,870000,Bali Tanjung Benoa & Uluwatu Tour Package From...,both,"Tanjung Benoa, Bali, Indonesia","Tanjung Benoa, Gang Guntur, Benoa, Kuta Selata...",-8.7657398,115.2232562,"{'natural': 'Tanjung Benoa', 'road': 'Gang Gun...",both
6,Bali Tanjung Benoa & Uluwatu Tour Package,Uluwatu,870000,Bali Tanjung Benoa & Uluwatu Tour Package From...,both,"Uluwatu Temple, Bali, Indonesia","Pura Luhur Uluwatu, Jalan Raya Uluwatu, Pecatu...",-8.8293693,115.0843428,"{'amenity': 'Pura Luhur Uluwatu', 'road': 'Jal...",both
7,Bali Kintamani Volcano View Tour Package,Kintamani,715000,Bali Kintamani Volcano View Tour Package From ...,both,"Kintamani, Bali, Indonesia","Kintamani, Bangli, Bali, Indonesia",-8.2495163,115.3482233,"{'town': 'Kintamani', 'region': 'Bangli', 'sta...",both


In [ ]:
print("Jumlah baris:", len(df_hasil_merge))

# Periksa duplikasi berdasarkan kolom penting
kolom_cek = [
    "nama_paket",
    "destinasi_key",
    "harga",
    "nama_lokasi",
    "latitude",
    "longitude"
]

print(
    "Duplikasi data:",
    df_hasil_merge.duplicated(
        subset=kolom_cek
    ).sum()
)

print(
    "Destinasi yang tidak ditemukan:",
    df_hasil_merge["nama_lokasi"].isna().sum()
)

print("\nStatus merge API:")
print(df_hasil_merge["status_api"].value_counts())

Jumlah baris: 8
Duplikasi data: 0
Destinasi yang tidak ditemukan: 0

Status merge API:
status_api
both          8
left_only     0
right_only    0
Name: count, dtype: int64


In [ ]:
import os

os.makedirs("data/process", exist_ok=True)

df_hasil_merge.to_csv(
    "data/process/hasil_merge_paket_destinasi.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan!")

File berhasil disimpan!


In [ ]:
os.makedirs("data/raw/api", exist_ok=True)

df_api.to_json(
    "data/raw/api/hasil_api_nominatim.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("Data API berhasil disimpan!")

Data API berhasil disimpan!


In [ ]:
import os

print("File hasil merge:",
      os.path.exists("data/process/hasil_merge_paket_destinasi.csv"))

print("File data API:",
      os.path.exists("data/raw/api/hasil_api_nominatim.json"))

File hasil merge: True
File data API: True


In [ ]:
import json
import os
from datetime import date

metadata_merge = {
    "nama_file": "hasil_merge_paket_destinasi.csv",
    "tanggal_proses": str(date.today()),
    "sumber_data": [
        {
            "nama": "Day Bali Tour",
            "jenis": "Web Scraping",
            "file": "hasil_scraping_daybalitour.csv"
        },
        {
            "nama": "OpenStreetMap Nominatim",
            "jenis": "API",
            "file": "hasil_api_nominatim.json",
            "url": "https://nominatim.openstreetmap.org/search"
        }
    ],
    "metode_penggabungan": "Pandas Merge",
    "kunci_penggabungan": "destinasi_key",
    "jenis_merge": "left",
    "validasi": "many_to_one",
    "jumlah_baris": int(len(df_hasil_merge)),
    "jumlah_kolom": int(len(df_hasil_merge.columns)),
    "jumlah_duplikasi": int(
        df_hasil_merge.duplicated(
            subset=[
                "nama_paket",
                "destinasi_key",
                "harga",
                "nama_lokasi",
                "latitude",
                "longitude"
            ]
        ).sum()
    ),
    "atribusi": "© OpenStreetMap contributors"
}

os.makedirs("docs", exist_ok=True)

with open("docs/metadata_merge.json", "w", encoding="utf-8") as file:
    json.dump(metadata_merge, file, indent=4, ensure_ascii=False)

print("Metadata berhasil disimpan!")

Metadata berhasil disimpan!


In [ ]:
with open("docs/metadata_merge.json", "r", encoding="utf-8") as file:
    metadata_cek = json.load(file)

print(json.dumps(metadata_cek, indent=4, ensure_ascii=False))

{
    "nama_file": "hasil_merge_paket_destinasi.csv",
    "tanggal_proses": "2026-11-25",
    "sumber_data": [
        {
            "nama": "Day Bali Tour",
            "jenis": "Web Scraping",
            "file": "hasil_scraping_daybalitour.csv"
        },
        {
            "nama": "OpenStreetMap Nominatim",
            "jenis": "API",
            "file": "hasil_api_nominatim.json",
            "url": "https://nominatim.openstreetmap.org/search"
        }
    ],
    "metode_penggabungan": "Pandas Merge",
    "kunci_penggabungan": "destinasi_key",
    "jenis_merge": "left",
    "validasi": "many_to_one",
    "jumlah_baris": 8,
    "jumlah_kolom": 11,
    "jumlah_duplikasi": 0,
    "atribusi": "© OpenStreetMap contributors"
}


In [ ]:
import pandas as pd

df_provenance = pd.read_csv("docs/provenance.csv")

print("Kolom provenance:")
print(df_provenance.columns.tolist())

print("\nData provenance saat ini:")
display(df_provenance.tail())

Kolom provenance:
['nama_sumber', 'url', 'tanggal_akses', 'lisensi', 'md5']

Data provenance saat ini:


,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783
4,day_bali_tour.html,https://www.daybalitour.com/best-bali-day-tour...,2026-11-24,Belum dinyatakan pada halaman sumber,a2d4750647a51c5252cb8b8bf0d28996


In [ ]:
import hashlib
import os

# Lokasi file hasil API
file_api = "data/raw/api/hasil_api_nominatim.json"

# Hitung MD5 file API
with open(file_api, "rb") as file:
    md5_api = hashlib.md5(file.read()).hexdigest()

# Data sumber API
data_api_provenance = {
    "nama_sumber": "OpenStreetMap Nominatim API",
    "url": "https://nominatim.openstreetmap.org/search",
    "tanggal_akses": "2026-11-25",
    "lisensi": "OpenStreetMap data: ODbL 1.0 (atribusi diperlukan)",
    "md5": md5_api
}

# Hindari penambahan baris API yang sama berulang kali
if not (
    df_provenance["nama_sumber"] == "OpenStreetMap Nominatim API"
).any():
    df_provenance = pd.concat(
        [df_provenance, pd.DataFrame([data_api_provenance])],
        ignore_index=True
    )

# Simpan kembali provenance
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Provenance berhasil diperbarui!")
display(df_provenance)

Provenance berhasil diperbarui!


,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783
4,day_bali_tour.html,https://www.daybalitour.com/best-bali-day-tour...,2026-11-24,Belum dinyatakan pada halaman sumber,a2d4750647a51c5252cb8b8bf0d28996
5,OpenStreetMap Nominatim API,https://nominatim.openstreetmap.org/search,2026-11-25,OpenStreetMap data: ODbL 1.0 (atribusi diperlu...,c6c90ca8bb18ec5d0395d4d5d4b0ebd0


In [ ]:
print("Total sumber data:", len(df_provenance))

print("\nJumlah berdasarkan jenis sumber:")
print(df_provenance["nama_sumber"])

Total sumber data: 6

Jumlah berdasarkan jenis sumber:
0    Jumlah Kunjungan Wisatawan Mancanegara per bul...
1    Jumlah Kunjungan Wisatawan Mancanegara per bul...
2    Jumlah Kunjungan Wisatawan Mancanegara per bul...
3    Jumlah Kunjungan Wisatawan Mancanegara per bul...
4                                   day_bali_tour.html
5                          OpenStreetMap Nominatim API
Name: nama_sumber, dtype: object


In [ ]:
import os

for folder, subfolder, files in os.walk("."):
    if ".config" in folder or "sample_data" in folder:
        continue

    for nama_file in files:
        if nama_file.endswith((".csv", ".json", ".html", ".md", ".txt", ".ipynb")):
            lokasi = os.path.join(folder, nama_file)
            print(lokasi)

./docs/provenance.csv
./docs/data_ethics_checklist.csv
./docs/metadata_merge.json
./docs/evidence/robots_daybalitour.txt
./docs/evidence/catatan_terms_daybalitour.txt
./drive/MyDrive/covid19.csv
./drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb
./drive/MyDrive/Colab Notebooks/praktikum3.ipynb
./drive/MyDrive/Colab Notebooks/Untitled0.ipynb
./drive/MyDrive/Colab Notebooks/ProjectArray.ipynb
./drive/MyDrive/Colab Notebooks/Untitled1.ipynb
./drive/MyDrive/Colab Notebooks/Untitled2.ipynb
./drive/MyDrive/Colab Notebooks/Untitled3.ipynb
./drive/MyDrive/Colab Notebooks/cuaca meteo.ipynb
./drive/MyDrive/Colab Notebooks/Untitled4.ipynb
./drive/MyDrive/Colab Notebooks/Open_Meteo.ipynb
./drive/MyDrive/Colab Notebooks/Untitled5.ipynb
./drive/MyDrive/Colab Notebooks/Salinan dari UTS_Data_Acquisition.ipynb
./data/process/hasil_download_gabungan.csv
./data/process/hasil_scraping.csv
./data/process/hasil_download_final.csv
./data/process/hasil_scraping.json
./data/process/hasil_download_bersi

In [ ]:
import os
import shutil

os.makedirs("notebooks", exist_ok=True)

sumber = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"

tujuan = "notebooks/UTS_Data_Acquisition.ipynb"

shutil.copy2(sumber, tujuan)

print("Notebook berhasil disalin!")
print("File tersedia:", os.path.exists(tujuan))

Notebook berhasil disalin!
File tersedia: True


In [ ]:
readme = """# Data Acquisition Management - UTS

## Deskripsi
Proyek ini dibuat untuk memenuhi tugas UTS
mata kuliah Akuisisi dan Manajemen Data.

## Sumber Data
1. BPS Indonesia (CSV)
2. Day Bali Tour (Web Scraping)
3. OpenStreetMap Nominatim (API)

## Struktur Data
- data/raw/download: Data CSV BPS
- data/raw/scraping: Data HTML hasil scraping
- data/raw/api: Data hasil API
- data/process: Data hasil pengolahan
- docs: Dokumentasi dan metadata
- notebooks: Notebook pengerjaan

## Hasil Pengolahan
Data BPS dibersihkan dan digabung.
Data scraping dan API digabung menggunakan
Pandas merge dengan validasi many_to_one.

## Dokumentasi
Informasi sumber data, metadata merge,
dan checklist etika data tersedia di folder docs.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README berhasil dibuat!")

README berhasil dibuat!


In [ ]:
print("README:", os.path.exists("README.md"))
print("Notebook:", os.path.exists(
    "notebooks/UTS_Data_Acquisition.ipynb"
))

README: True
Notebook: True


In [ ]:
print(os.path.exists("requirements.txt"))

True


In [ ]:
requirements = """pandas
requests
beautifulsoup4
openpyxl
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt berhasil dibuat!")

requirements.txt berhasil dibuat!


In [ ]:
import os

print("Lokasi kerja saat ini:")
print(os.getcwd())

print("\nIsi folder:")
print(os.listdir("."))

Lokasi kerja saat ini:
/content

Isi folder:
['.config', 'notebooks', 'requirements.txt', 'docs', 'drive', 'data', 'README.md', 'sample_data']


In [ ]:
!git --version

git version 2.43.0


In [ ]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [ ]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	README.md
	data/
	docs/
	drive/
	notebooks/
	requirements.txt
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git branch -m main

In [ ]:
isi_gitignore = """
.config/
drive/
sample_data/
__pycache__/
.ipynb_checkpoints/
"""

with open(".gitignore", "w") as file:
    file.write(isi_gitignore.strip() + "\n")

print(".gitignore berhasil dibuat!")

.gitignore berhasil dibuat!


In [ ]:
!git add README.md requirements.txt .gitignore data/ docs/ notebooks/

In [ ]:
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   data/process/hasil_download_bersih.csv
	new file:   data/process/hasil_download_final.csv
	new file:   data/process/hasil_download_gabungan.csv
	new file:   data/process/hasil_merge_paket_destinasi.csv
	new file:   data/process/hasil_scraping.csv
	new file:   data/process/hasil_scraping.json
	new file:   data/process/hasil_scraping_daybalitour.csv
	new file:   data/raw/api/hasil_api_nominatim.json
	new file:   data/raw/download/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
	new file:   data/raw/download/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
	new file:   data/raw/download/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
	new file:   da

In [174]:
!git config --global user.name "Ahmad Paesoel"
!git config --global user.email "kallenkaslana95@gmail.com"

In [175]:
!git commit -m "Initial commit - UTS Akuisisi dan Manajemen Data"

On branch main
nothing to commit, working tree clean


In [176]:
!git log -1 --oneline

6141f1d (HEAD -> main) Initial commit - UTS Akuisisi dan Manajemen Data


In [177]:
!git remote add origin https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS.git

In [178]:
!git remote -v

origin	https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS.git (fetch)
origin	https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS.git (push)


In [179]:
!git ls-remote --heads origin

4ee60f287b9199233f0b2fe9257d0f8ecc228cb1	refs/heads/main


In [180]:
!git fetch origin

remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 68 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (68/68), 72.97 KiB | 655.00 KiB/s, done.
From https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS
 * [new branch]      main       -> origin/main


In [181]:
!git status

On branch main
nothing to commit, working tree clean


In [182]:
!git log --oneline --decorate --all -5

6141f1d (HEAD -> main) Initial commit - UTS Akuisisi dan Manajemen Data
4ee60f2 (origin/main) Create .gitkeep
99fdbe9 Delete notebooks/UTS_Data_Acquisition.ipynb
ca9fd9f Delete notebooks/.gitkeep
4c103a6 Rename UTS_Data_Acquisition.ipynb to notebooks/UTS_Data_Acquisition.ipynb


In [183]:
!git ls-tree -r --name-only origin/main

README.md
data/process/.gitkeep
data/raw/.gitkeep
data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
docs/data_ethics_checklist.md
docs/evidence/.gitkeep
docs/provenance.csv
notebooks/.gitkeep
requirements.txt


In [184]:
!git merge origin/main --allow-unrelated-histories

Auto-merging README.md
CONFLICT (add/add): Merge conflict in README.md
Auto-merging docs/provenance.csv
CONFLICT (add/add): Merge conflict in docs/provenance.csv
Automatic merge failed; fix conflicts and then commit the result.


In [185]:
!git status

On branch main
You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Changes to be committed:
	new file:   data/process/.gitkeep
	new file:   data/raw/.gitkeep
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
	new file:   docs/data_ethics_checklist.md
	new file:   docs/evidence/.gitkeep
	new file:   notebooks/.gitkeep

Unmerged paths:
  (use "git add <file>..." to mark resolution)
	both added:      README.md
	both added:      docs/provenance.csv



In [186]:
!cat README.md

<<<<<<< HEAD
# Data Acquisition Management - UTS

## Deskripsi
Proyek ini dibuat untuk memenuhi tugas UTS
mata kuliah Akuisisi dan Manajemen Data.

## Sumber Data
1. BPS Indonesia (CSV)
2. Day Bali Tour (Web Scraping)
3. OpenStreetMap Nominatim (API)

## Struktur Data
- data/raw/download: Data CSV BPS
- data/raw/scraping: Data HTML hasil scraping
- data/raw/api: Data hasil API
- data/process: Data hasil pengolahan
- docs: Dokumentasi dan metadata
- notebooks: Notebook pengerjaan

## Hasil Pengolahan
Data BPS dibersihkan dan digabung.
Data scraping dan API digabung menggunakan
Pandas merge dengan validasi many_to_one.

## Dokumentasi
Informasi sumber data, metadata merge,
dan checklist etika data tersedia di folder docs.
# Data_Acqusition_Management_UTS
# UTS Data Acquisition & Management

## 1. Identitas Mahasiswa
- Nama: [Nama lengkap]
- NIM: [NIM]
- Kelas: F
- Program Studi: Informatika

## 2. Judul Proyek
Pengumpulan Data Paket Wisata Bali

## 3. Tujuan
Proyek ini bertujuan untuk me

In [189]:
isi_readme = """
# Data Acquisition Management - UTS

## 1. Identitas Mahasiswa
- Nama: [Ahmad Paesoel]
- NIM: [2501010143]
- Kelas: F
- Program Studi: Informatika

## 2. Deskripsi Proyek
Proyek ini dibuat untuk memenuhi tugas UTS mata kuliah Akuisisi dan Manajemen Data.
Proyek berfokus pada pengumpulan, pemeriksaan, pengolahan, dan penggabungan data
yang berkaitan dengan pariwisata dan paket wisata di Bali.

## 3. Sumber Data
1. BPS Indonesia (dataset CSV)
2. Day Bali Tour (web scraping)
3. OpenStreetMap Nominatim (API)

## 4. Metode Akuisisi
- Mengunduh dataset CSV dari BPS.
- Melakukan web scraping menggunakan Python dan BeautifulSoup.
- Mengambil data lokasi menggunakan API Nominatim.

## 5. Tools yang Digunakan
- Python
- Google Colab
- Pandas
- Requests
- BeautifulSoup
- Git dan GitHub

## 6. Struktur Proyek
- data/raw/download/: Data CSV BPS.
- data/raw/scraping/: Data HTML hasil scraping.
- data/raw/api/: Data hasil API.
- data/process/: Data hasil pengolahan.
- docs/: Dokumentasi, provenance, metadata, dan checklist etika data.
- notebooks/: Notebook pengerjaan.

## 7. Hasil Pengolahan
Data BPS dibersihkan dan digabungkan.
Data paket wisata hasil scraping dan data lokasi dari API
digabungkan menggunakan Pandas merge dengan validasi many_to_one.

Hasil penggabungan diperiksa untuk memastikan tidak terdapat
duplikasi baris dan data lokasi yang hilang.

## 8. Dokumentasi
Informasi sumber data, metadata penggabungan, dan checklist etika data
tersedia di folder docs.

## 9. Catatan
Data mentah disimpan tanpa perubahan. Proses pembersihan dan pengolahan
dilakukan pada salinan data agar sumber aslinya tetap terjaga.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(isi_readme)

print("README berhasil diperbarui.")

README berhasil diperbarui.


In [190]:
!git add README.md

In [191]:
!cat docs/provenance.csv

<<<<<<< HEAD
﻿nama_sumber,url,tanggal_akses,lisensi,md5
"Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv",https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
"Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv",https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
"Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv",https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
"Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Ind

In [192]:
import pandas as pd

data_provenance = [
    {
        "nama_sumber": "BPS 2022",
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": "daba6b473fdf84e7d075cc32e7c22122"
    },
    {
        "nama_sumber": "BPS 2023",
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": "41bd665fdc69490698b230d73c099d94"
    },
    {
        "nama_sumber": "BPS 2024",
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": "871c2d3afa1f43fd7aa02bb116a120fb"
    },
    {
        "nama_sumber": "BPS 2025",
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": "0a25d32a1fc15e65974727e20f22a783"
    },
    {
        "nama_sumber": "Day Bali Tour",
        "url": "https://www.daybalitour.com/best-bali-day-tour-packages/",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": "a2d4750647a51c5252cb8b8bf0d28996"
    },
    {
        "nama_sumber": "OpenStreetMap Nominatim API",
        "url": "https://nominatim.openstreetmap.org/search",
        "tanggal_akses": "2026-11-25",
        "lisensi": "OpenStreetMap data: ODbL 1.0 (atribusi diperlukan)",
        "md5": "c6c90ca8bb18ec5d0395d4d5d4b0ebd0"
    }
]

df_provenance = pd.DataFrame(data_provenance)

df_provenance.to_csv(
    "docs/provenance.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_provenance)
print("\nJumlah sumber:", len(df_provenance))

                   nama_sumber  \
0                     BPS 2022   
1                     BPS 2023   
2                     BPS 2024   
3                     BPS 2025   
4                Day Bali Tour   
5  OpenStreetMap Nominatim API   

                                                 url tanggal_akses  \
0  https://www.bps.go.id/id/statistics-table/2/MT...    2026-11-24   
1  https://www.bps.go.id/id/statistics-table/2/MT...    2026-11-24   
2  https://www.bps.go.id/id/statistics-table/2/MT...    2026-11-24   
3  https://www.bps.go.id/id/statistics-table/2/MT...    2026-11-24   
4  https://www.daybalitour.com/best-bali-day-tour...    2026-11-24   
5         https://nominatim.openstreetmap.org/search    2026-11-25   

                                             lisensi  \
0               Belum dinyatakan pada halaman sumber   
1               Belum dinyatakan pada halaman sumber   
2               Belum dinyatakan pada halaman sumber   
3               Belum dinyatakan pada halaman 

In [193]:
!git status

On branch main
You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Changes to be committed:
	modified:   README.md
	new file:   data/process/.gitkeep
	new file:   data/raw/.gitkeep
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
	new file:   docs/data_ethics_checklist.md
	new file:   docs/evidence/.gitkeep
	new file:   notebooks/.gitkeep

Unmerged paths:
  (use "git add <file>..." to mark resolution)
	both added:      docs/provenance.csv



In [194]:
!git add docs/provenance.csv

In [195]:
!git status

On branch main
All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)

Changes to be committed:
	modified:   README.md
	new file:   data/process/.gitkeep
	new file:   data/raw/.gitkeep
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
	new file:   data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
	new file:   docs/data_ethics_checklist.md
	new file:   docs/evidence/.gitkeep
	modified:   docs/provenance.csv
	new file:   notebooks/.gitkeep



In [196]:
!git commit -m "Merge GitHub repository with UTS project"

[main a8320f7] Merge GitHub repository with UTS project


In [197]:
!git status

On branch main
nothing to commit, working tree clean


In [198]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [200]:
from google.colab import userdata
import os
import subprocess
import tempfile

token = userdata.get("GITHUB_TOKEN")

askpass = tempfile.NamedTemporaryFile(
    mode="w",
    delete=False
)

askpass.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

askpass.close()
os.chmod(askpass.name, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass.name
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    subprocess.run(
        ["git", "push", "origin", "main"],
        env=env,
        check=True
    )
finally:
    os.remove(askpass.name)

In [201]:
!git status

On branch main
nothing to commit, working tree clean


In [202]:
!git log -1 --oneline

a8320f7 (HEAD -> main, origin/main) Merge GitHub repository with UTS project


In [203]:
!git ls-remote --heads origin main

a8320f7f052f2e1546c4223086376470b054f2ae	refs/heads/main


In [204]:
import os

for folder, subfolder, files in os.walk("data/raw"):
    print(f"\nFolder: {folder}")
    for file in files:
        print("  -", file)


Folder: data/raw
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
  - .gitkeep
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv

Folder: data/raw/download
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
  - Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv

Folder: data/raw/api
  - hasil_api_no

In [205]:
import os
import hashlib

for tahun in ["2022", "2023", "2024", "2025"]:
    nama_file = next(
        f for f in os.listdir("data/raw")
        if tahun in f and f.endswith(".csv")
    )

    file_utama = os.path.join("data/raw", nama_file)
    file_download = os.path.join("data/raw/download", nama_file)

    def md5_file(path):
        with open(path, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()

    print(
        tahun,
        "Identik:",
        md5_file(file_utama) == md5_file(file_download)
    )

2022 Identik: True
2023 Identik: True
2024 Identik: True
2025 Identik: True


In [206]:
import os

for tahun in ["2022", "2023", "2024", "2025"]:
    for nama_file in os.listdir("data/raw"):
        if tahun in nama_file and nama_file.endswith(".csv"):
            path = os.path.join("data/raw", nama_file)
            os.remove(path)
            print("Dihapus:", nama_file)

print("\nSalinan tambahan berhasil dibersihkan.")

Dihapus: Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
Dihapus: Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
Dihapus: Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
Dihapus: Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv

Salinan tambahan berhasil dibersihkan.


In [207]:
!git add -A
!git status

On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
	deleted:    data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
	deleted:    data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
	deleted:    data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv



In [208]:
!git commit -m "Clean duplicate raw CSV files"

[main aab47b7] Clean duplicate raw CSV files
 4 files changed, 156 deletions(-)
 delete mode 100644 data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
 delete mode 100644 data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
 delete mode 100644 data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
 delete mode 100644 data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv


In [212]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [240]:
from google.colab import userdata
import os
import subprocess
import tempfile

token = userdata.get("GITHUB_TOKEN")

askpass = tempfile.NamedTemporaryFile(
    mode="w",
    delete=False
)

askpass.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

askpass.close()
os.chmod(askpass.name, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass.name
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    hasil = subprocess.run(
        ["git", "push", "origin", "main"],
        env=env,
        check=True,
        capture_output=True,
        text=True
    )
    print(hasil.stdout)
    print(hasil.stderr)
    print("Push berhasil!")
except subprocess.CalledProcessError as e:
    print("Push gagal:")
    print(e.stderr)
finally:
    os.remove(askpass.name)


To https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS.git
   2d713c3..63dbfd9  main -> main

Push berhasil!


In [214]:
!git fetch origin

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 8 (delta 4), reused 1 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.95 KiB | 500.00 KiB/s, done.
From https://github.com/Ahmadlazz/Data_Acqusition_Management_UTS
   a8320f7..896578b  main       -> origin/main


In [215]:
!git log --oneline --graph --decorate --all -10

* aab47b7 (HEAD -> main) Clean duplicate raw CSV files
| * 896578b (origin/main) Delete data/process/.gitkeep
| * b2eabd1 Delete data/raw/.gitkeep
|/  
*   a8320f7 Merge GitHub repository with UTS project
|\  
| * 4ee60f2 Create .gitkeep
| * 99fdbe9 Delete notebooks/UTS_Data_Acquisition.ipynb
| * ca9fd9f Delete notebooks/.gitkeep
| * 4c103a6 Rename UTS_Data_Acquisition.ipynb to notebooks/UTS_Data_Acquisition.ipynb
| * d58bc52 Dibuat menggunakan Colab
| * 1791d34 Delete UTS_Data_Acquisition.ipynb


In [216]:
!git status

On branch main
nothing to commit, working tree clean


In [217]:
!git merge origin/main -m "Merge latest GitHub changes"

Merge made by the 'ort' strategy.
 data/process/.gitkeep | 1 -
 data/raw/.gitkeep     | 1 -
 2 files changed, 2 deletions(-)
 delete mode 100644 data/process/.gitkeep
 delete mode 100644 data/raw/.gitkeep


In [218]:
!git status

On branch main
nothing to commit, working tree clean


In [220]:
import os

file_asli = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"
file_proyek = "/content/notebooks/UTS_Data_Acquisition.ipynb"

print("Notebook Drive:", os.path.exists(file_asli))
print("Notebook proyek:", os.path.exists(file_proyek))

Notebook Drive: True
Notebook proyek: True


In [221]:
import os
import hashlib

file_asli = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"
file_proyek = "/content/notebooks/UTS_Data_Acquisition.ipynb"

def cek_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

print("Ukuran notebook Drive:", os.path.getsize(file_asli))
print("Ukuran notebook proyek:", os.path.getsize(file_proyek))

print("\nHash Drive:", cek_hash(file_asli))
print("Hash proyek:", cek_hash(file_proyek))

print("\nApakah identik:", cek_hash(file_asli) == cek_hash(file_proyek))

Ukuran notebook Drive: 647848
Ukuran notebook proyek: 585204

Hash Drive: c49cac979d083e7d90f78792cfa3ad07
Hash proyek: c4457433a1a1d28f337bdc6fff13b08f

Apakah identik: False


In [222]:
import json

with open(file_asli, "r", encoding="utf-8") as f:
    notebook_drive = json.load(f)

with open(file_proyek, "r", encoding="utf-8") as f:
    notebook_proyek = json.load(f)

print("Notebook Google Drive:")
print("Jumlah sel:", len(notebook_drive["cells"]))

print("\nNotebook folder proyek:")
print("Jumlah sel:", len(notebook_proyek["cells"]))

print("\nJumlah sel kode Drive:",
      sum(1 for c in notebook_drive["cells"] if c["cell_type"] == "code"))

print("Jumlah sel kode proyek:",
      sum(1 for c in notebook_proyek["cells"] if c["cell_type"] == "code"))

Notebook Google Drive:
Jumlah sel: 168

Notebook folder proyek:
Jumlah sel: 114

Jumlah sel kode Drive: 147
Jumlah sel kode proyek: 93


In [223]:
import shutil
import os

file_asli = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"
file_proyek = "/content/notebooks/UTS_Data_Acquisition.ipynb"

# Cadangkan notebook proyek lama
shutil.copy2(
    file_proyek,
    file_proyek + ".backup"
)

# Salin notebook terbaru dari Google Drive
shutil.copy2(
    file_asli,
    file_proyek
)

print("Notebook berhasil diperbarui!")
print("Cadangan notebook lama juga tersedia.")

Notebook berhasil diperbarui!
Cadangan notebook lama juga tersedia.


In [224]:
print("Apakah notebook identik:",
      cek_hash(file_asli) == cek_hash(file_proyek))

Apakah notebook identik: True


In [225]:
!git add notebooks/UTS_Data_Acquisition.ipynb

In [228]:
!git commit -m "Update notebook with latest UTS work"

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/UTS_Data_Acquisition.ipynb.backup

nothing added to commit but untracked files present (use "git add" to track)


In [230]:
%cd /content

!git status

/content
On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/UTS_Data_Acquisition.ipynb.backup

nothing added to commit but untracked files present (use "git add" to track)


In [231]:
import os

print(os.path.getmtime(
    "/content/notebooks/UTS_Data_Acquisition.ipynb"
))

1795598808.0


In [232]:
import hashlib

file_project = "/content/notebooks/UTS_Data_Acquisition.ipynb"

with open(file_project, "rb") as f:
    hash_project = hashlib.md5(f.read()).hexdigest()

print("Hash notebook project:", hash_project)

Hash notebook project: c6c6823fa952f64901a79ae402465282


In [233]:
file_drive = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"

with open(file_drive, "rb") as f:
    hash_drive = hashlib.md5(f.read()).hexdigest()

print("Hash Google Drive :", hash_drive)
print("Hash Project      :", hash_project)
print("Apakah sama?      :", hash_drive == hash_project)

Hash Google Drive : 2db4f425d5844709b7194fbdf98731fe
Hash Project      : c6c6823fa952f64901a79ae402465282
Apakah sama?      : False


In [234]:
import shutil

file_drive = "/content/drive/MyDrive/Colab Notebooks/UTS_Data_Acquisition.ipynb"
file_project = "/content/notebooks/UTS_Data_Acquisition.ipynb"

shutil.copy2(file_drive, file_project)

print("Notebook berhasil disalin!")

Notebook berhasil disalin!


In [235]:
with open(file_drive, "rb") as f:
    hash_drive = hashlib.md5(f.read()).hexdigest()

with open(file_project, "rb") as f:
    hash_project = hashlib.md5(f.read()).hexdigest()

print("Apakah sama?", hash_drive == hash_project)

Apakah sama? True


In [236]:
%cd /content
!git status

/content
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/UTS_Data_Acquisition.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/UTS_Data_Acquisition.ipynb.backup

no changes added to commit (use "git add" and/or "git commit -a")


In [237]:
!git add notebooks/UTS_Data_Acquisition.ipynb

In [238]:
!git commit -m "Hapus cell kode contoh yang tidak digunakan"

[main 63dbfd9] Hapus cell kode contoh yang tidak digunakan
 1 file changed, 1 insertion(+), 1 deletion(-)


In [239]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address
